In [1]:
from agents import Agent, OpenAIChatCompletionsModel, Runner, trace, WebSearchTool
from bs4 import BeautifulSoup
from datetime import datetime
from dotenv import load_dotenv
from workflow.common import extract_html
import gspread
from IPython.display import display, Markdown
import json
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, SummaryIndex
from llama_index.llms.openai import OpenAI
from llama_index.llms.openrouter import OpenRouter
from llama_index.core import Settings
from mailjet_rest import Client
from openai import OpenAI, AsyncOpenAI
import os
import requests
import pandas as pd
from pydantic import BaseModel, Field
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import tempfile
import time
from tqdm import tqdm
from webdriver_manager.chrome import ChromeDriverManager

load_dotenv(override=True)

C:\Users\User\anaconda3\Lib\site-packages\torch\utils\_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


True

In [2]:
openai_api_key = os.getenv('OPENAI_API_KEY')
openai_base_url = os.getenv('BASE_URL')

client = AsyncOpenAI(
    base_url=openai_base_url,
    api_key=openai_api_key,
)


In [3]:
model = OpenAIChatCompletionsModel(model="openai/gpt-4o-mini", openai_client=client)
model_4o = OpenAIChatCompletionsModel(model="openai/gpt-4o:online", openai_client=client)
model_anthropic_online = OpenAIChatCompletionsModel(model="anthropic/claude-3.7-sonnet:online", openai_client=client)

In [4]:
llm = OpenRouter(
    api_key=openai_api_key,
    model="openai/gpt-4o-mini",        # Supports 128k context
    max_tokens=4000,                   # Allow long summaries
    context_window=128000,             # ← THIS IS THE KEY FIX
    temperature=0.1,
    timeout=600,                       # Prevent timeouts on large docs
    max_retries=3,
)

# llm = OpenRouter(
#     api_key=openai_api_key,
#     model="anthropic/claude-3-haiku",  # 200k context → solves everything
#     max_tokens=4000,
#     temperature=0.1,
#     timeout=900,
# )

Settings.llm = llm

Settings.context_window = 128000   # ← THIS FIXES THE NEGATIVE CONTEXT ERROR
Settings.chunk_size = 8192         # Larger chunks = fewer calls
Settings.chunk_overlap = 200

In [5]:
response = await client.chat.completions.create(
#     model="openai/gpt-4.1-nano",
    model = "qwen/qwen-2.5-72b-instruct",
    messages=[{"role": "user", "content": "Hi there."}],
)

print(response.choices[0].message.content.strip())

Hello! How can I assist you today?


In [4]:
spreadsheet_id = "1oHKGMuBynXOJkkQpDTAtjfsv-jrTXpzI2jj29VCCDaM"
sheet_gid = "0"
url = f"https://docs.google.com/spreadsheets/d/{spreadsheet_id}/export?format=csv&gid={sheet_gid}"
df = pd.read_csv(url)
df.head()

,Country,News,Website,Date,Headline,Link,Summary,Comment,Method
0,Afghanistan,Pajhwok Afghan News,https://www.pajhwok.com,9/12/2025,Doha Forum stresses Afghanistan’s role in regi...,https://pajhwok.com/2025/12/08/doha-forum-stre...,- The Doha Forum emphasized Afghanistan's pote...,NaN,NaN
1,Albania,Gazeta Panorama,https://www.panorama.com.al/,9/12/2025,"""We have not been divided!"" - The meetings of ...",https://www.panorama.com.al/skemi-qene-te-ndar...,- Salianji's meetings with Democratic Party me...,NaN,NaN
2,Algeria,El Khabar,https://www.elkhabar.com/fr,9/12/2025,The President of the Republic presides over a ...,https://www.elkhabar.com/fr/nation/le-presiden...,- The President of the Republic leads a meetin...,NaN,NaN
3,Andorra,Bondia.ad,https://www.bondia.ad/,9/12/2025,"Grandvalira closes the bridge with nearly 47,0...",https://www.bondia.ad/societat/grandvalira-tan...,- Grandvalira ski resort has successfully clos...,NaN,NaN
4,Angola,Jornal de Angola (state-owned),http://www.jornaldeangola.ao,9/12/2025,NaN,NaN,NaN,NaN,NaN


In [5]:
global_headlines_json = df.to_json(orient='records')

In [6]:
credentials_file = "global-headlines-474905-9494f258e0a5.json"
gc = gspread.service_account(filename=credentials_file)
spreadsheet = gc.open_by_key(spreadsheet_id)
worksheet = spreadsheet.worksheet("Sheet1")

In [5]:
def remove_html_tags(html_content):
    soup = BeautifulSoup(html_content, "html.parser")
    return soup.get_text()

In [6]:
website = "https://asia.nikkei.com/business/tech/semiconductors/tsmc-turns-japan-into-3rd-advanced-chip-base-as-ai-demand-soars/"

In [7]:
html_dict = extract_html.get_raw_html(website)
html_string = html_dict["html"]
clean_text = remove_html_tags(html_string)

if len(html_string) > 200000:
    html_string_input = clean_text
else:
    html_string_input = html_string

In [8]:
html_string_input

"\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n   TSMC turns Japan into 3rd advanced chip base as AI demand soars - Nikkei Asia\n  \n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n        World\n       \n\n\n\n\n\n\n\n\n           China\n          \n\n\n\n\n\n\n\n           Japan\n          \n\n\n\n\n\n\n\n           India\n          \n\n\n\n\n\n\n\n           South Korea\n          \n\n\n\n\n\n\n\n           Indonesia\n          \n\n\n\n\n\n\n\n           Taiwan\n          \n\n\n\n\n\n\n\n           Thailand\n          \n\n\n\n\n\n\n\n           U.S.\n          \n\n\n\n\n\n\n\n\n\n           East Asia\n           \n\n\n\n\n\n\n\n\n              China\n             \n\n\n\n\n\n\n\n              Hong Kong\n             \n\n\n\n\n\n\

In [39]:
prompt = f'''
You are good at reading html text, visualize the content, and identify the headline of the day.
Given this HTML text from a news website: {html_string_input}

CRITICAL REQUIREMENTS:
1. Identify the main headline of the day
2. ALWAYS translate to English if the headline is in any other language
3. Output ONLY the English headline text, nothing else
4. Do not include any non-English words or phrases
5. Ensure the headline is in proper English grammar and spelling
6. If the headline contains any non-English characters or words, translate them to English

IMPORTANT: The output must be 100% in English. If you cannot translate a headline to English, output "No headline found".
'''

response = await client.chat.completions.create(
#     model="anthropic/claude-3.7-sonnet",
    model="openai/gpt-4o-mini",
#     model="qwen/qwen2.5-vl-32b-instruct",
    messages=[{"role": "user", "content": prompt}],
)

headline = response.choices[0].message.content.strip()
print(headline)

"Quarter by quarter: What are the property prices in Sofia at the beginning of 2026?"


In [236]:
print(len(html_string))
print(len(clean_text))

134808
20743


In [237]:
# html_string

In [238]:
soup = BeautifulSoup(html_string, "html.parser")

# Extract all links
links = {a.get_text(): a for a in soup.find_all('a')}

# Get text without HTML tags
# text = soup.get_text()
links_on_page = ""

# Append links to the text
for link_text, url in links.items():
    links_on_page += f" [{link_text}]({url})"

In [239]:
links_on_page

' [\n     IMG\n    ](<a href="https://www.ibg.bg/?utm_source=dnes&amp;utm_medium=link&amp;utm_campaign=lenta" rel="noopener" style="background-image: url(https://automedia.investor.bg/media/files/uploadedfiles/6e472c88bea7190ba9fda2ff6674e225-img-logo.png);     position: relative;  top: 3px;    width: 34px;     height: 13px;     overflow: hidden;     display: inline-block;     padding: 0;     text-indent: -9999px; background-repeat: no-repeat;     padding-right: 5px;" target="_blank">\n     IMG\n    </a>) [\n     Investor\n    ](<a href="https://www.investor.bg/?utm_source=dnes&amp;utm_medium=link&amp;utm_campaign=lenta" rel="noopener" style="" target="_blank">\n     Investor\n    </a>) [\n     Dnes\n    ](<a href="https://www.dnes.bg/?utm_source=dnes&amp;utm_medium=link&amp;utm_campaign=lenta" rel="noopener" style="" target="_blank">\n     Dnes\n    </a>) [\n     Bloombergtv\n    ](<a href="https://www.bloombergtv.bg/?utm_source=dnes&amp;utm_medium=link&amp;utm_campaign=lenta" rel="no

In [240]:
headline = response.choices[0].message.content.strip()
headline

'Arrests made for the rampage near the DPS headquarters'

In [241]:
prompt = f'''
Given this identified headline on a news website: "{headline}"
Note that the headline is being translated into English if the original text is non-English.
Please check if the link to that headline is in {links_on_page[:150000]}
If so, please output only the link. 
Note that sometime only the relative url is included. If so, you need to output the absolute url with the root website {website}.
If no link is founded, just output 'N/A'
Do not output anything else other than the link
Do not output any html tags e.g., '<a href=', '</a>'
'''

response = await client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": prompt}
    ]
)

In [242]:
link = response.choices[0].message.content
link

'https://www.dnes.bg/a/1-bulgaria/701984-v-kadar-ima-zadarzhani-za-pogroma-kray-tsentralata-na-dps'

In [30]:
link = "http://www.aastocks.com/sc/mobile/news.aspx?newsid=NOW.1496320&newstype=71&newssource=AAFN"

In [31]:
html_dict = extract_html.get_raw_html(link)

In [32]:
html_string = html_dict["html"]
clean_text = remove_html_tags(html_string)
clean_text

'\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n   《大行》里昂：携程短期利润率或受压  料中国反垄断行动更常态化财经新闻 Financial News\n  \n\n\n\n\n\n\n\n\n\n\n       桌面版\n      \n\n\n\n\n\n\n\n\n\n\n\n        最新搜看股票\n        \n\n\n\n\n\n\n\n\n\n       报价\n      \n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n       实时行情\n      \n\n\n\n\n\n\n\n       市场\n      \n\n\n\n\n\n\n\n       新闻\n      \n\n\n\n\n\n\n\n       指数\n      \n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n          返回\n         \n\n\n         放大\xa0+\n        \n\n         缩小\xa0-\n        \n\n         重点新闻\n        \n\n\n\n\n\n         《大行》里昂：携程短期利润率或受压  料中国反垄断行动更常态化\n        \n\n\n\n\n\n\n\n             推荐\n            \n\n             14\n            \n\n\n\n\n\n\n\n             利好\n            \n\n             16\n            \n\n\n\n\n\n\n\n             利淡\n            \n\n             21\n            \n\n\n\n\n\n\n          AASTOCKS新闻\n         \n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n         里昂发表研究报告指，国家市场监管总局立案调查携程-S(09961.HK)垄断行为，目前公司持续正常营运，惟受此消息影响，携程(TC

In [33]:
# clean_text

In [35]:
prompt = f'''
You are a capable journalist. This is the content of a news: {clean_text}
Please extract the title of the news in its original language. Do not output anything else
'''

response = await client.chat.completions.create(
#     model="gpt-4o-mini",
    model="deepseek/deepseek-v3.2",
    messages=[
        {"role": "user", "content": prompt}
    ]
)

In [36]:
summary = response.choices[0].message.content
print(summary)

《大行》里昂：携程短期利润率或受压  料中国反垄断行动更常态化


In [ ]:
prompt = f'''
You are a capable journalist. This is the headline of today: {clean_text}
Please summarize in 2-3 English bullets to capture the key informaiton. Each bullet should be very concise.
You may assume your readers know nothing about the country's news and politics
The source may not be in English. But make sure the summary is in English
If the text doesn't seem to be a legit news article, just output 'N/A'. Do not make things up.
IMPORTANT: When referring to Donald Trump, always refer to him as "President Donald Trump" or "US President Donald Trump". He is the CURRENT President of the United States as of 2025. Do NOT label him as "Former President" or "Ex-President" - this is incorrect.
'''

response = await client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": prompt}
    ]
)

In [ ]:
remove_html_tags_keep_links(html_string)

In [ ]:
country = "Algeria"
website = df[df['Country'] == country]['Website'].values[0]
print(f"Website: {website}")

In [ ]:
def find_row_number(df, country):
    row_number = df.index[df['Country'] == country].tolist()
    return row_number[0] + 2 if row_number else None

row_number = find_row_number(df, country)
today = datetime.now().strftime("%d/%m/%Y")

worksheet.update_cell(row_number, 4, today)
worksheet.update_cell(row_number, 5, headline_info["headline"])
worksheet.update_cell(row_number, 6, headline_info["link"])

print(f"Updated {country}")


In [ ]:
countries = list(df["Country"])

for country in tqdm(countries[:5]):
    try:
        website = df[df['Country'] == country]['Website'].values[0]
        if pd.isna(website) or website == "":
            continue
        
        html_dict = extract_html.get_raw_html(website)
        if not html_dict:
            continue
        
        html_string = html_dict["html"]
        clean_text = remove_html_tags(html_string)
        
        prompt = f'''
        Given this HTML text: {clean_text}
        Identify the headline and link.
        Output as JSON with "headline" and "link" keys.
        '''
        
        response = client.beta.chat.completions.parse(
            model="openai/gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            response_format=Headline,
        )
        
        headline_info = json.loads(response.choices[0].message.content)
        
        row_number = find_row_number(df, country)
        today = datetime.now().strftime("%d/%m/%Y")
        
        worksheet.update_cell(row_number, 4, today)
        worksheet.update_cell(row_number, 5, headline_info["headline"])
        worksheet.update_cell(row_number, 6, headline_info["link"])
        
        print(f"✅ {country}: {headline_info['headline'][:50]}...")
        
    except Exception as e:
        print(f"❌ {country}: {str(e)}")


In [15]:
INSTRUCTIONS = (
f'''
You are a global news curator responsible for creating a comprehensive newsletter based on the latest headlines from every country. Your goal is to summarize the most significant news stories, focusing on global politics, major local developments, natural disasters, and other impactful events.

Gather Headlines: You will receive a list of headlines from various countries.

Prioritize Content: Identify and prioritize the most important news stories that have global relevance or significant local impact. You should also include news from lesser-known countries to give the readers a broader perspective.

Create an Outline:

Headlines around the globe - {datetime.now().strftime("%b %d, %Y")}
Daily news sourced directly from local newspapers

Regional Highlights:
Africa / Middle East: Key developments and notable events.
Americas: Focus on political changes and major local news.
Asia: Important updates on economic and political situations.
Europe: Significant political events and social movements.
Oceania: Relevant stories and developments.

Each news should be like:
- **[Israel-Hamas Ceasefire](https://www.haaretz.com/ty-tag/2023-israel-gaza-war-0000018b-1458-df09-a9db-5df99abd0000)**: A temporary ceasefire has been agreed between Israel and Hamas, aiming to facilitate humanitarian efforts and stabilize the region. (source of the news)\n

The first few words should accurately summarize what that news in about.
The classification of location should be based on where the event took place, not the location of the news source.

Draft the Newsletter: Write a detailed and engaging newsletter that:

Summarizes each headline clearly and concisely.
Provides context where necessary to enhance understanding. The country / newspaper publishing the news may be relevant.
Includes the sources in a link if available.
Maintains an organized flow for easy reading.
If needed, search the web for additional context / information. You may not assume the readers know the name of a political person or a city.
Do not invent your own facts. This is important.
Final Output: Ensure the newsletter is formatted in Markdown, aiming for 1500 words at least. The content should be well-structured and informative, catering to a global audience interested in current affairs.
'''
)

writer_agent = Agent(
    name="WriterAgent",
    instructions=INSTRUCTIONS,
    model=model_anthropic_online,
)

In [7]:
result = await Runner.run(writer_agent, global_headlines_json)

[non-fatal] Tracing client error 403: {"error":{"code":"unsupported_country_region_territory","message":"Country, region, or territory not supported","param":null,"type":"request_forbidden"}}
[non-fatal] Tracing client error 403: {"error":{"code":"unsupported_country_region_territory","message":"Country, region, or territory not supported","param":null,"type":"request_forbidden"}}


In [8]:
display(Markdown(result.final_output))

# Headlines around the globe - Nov 02, 2025
## Daily news sourced directly from local newspapers

### Regional Highlights

#### Africa / Middle East
- **[Trump Threatens Nigeria](https://www.krone.at/3945418)**: Former US President Donald Trump has issued threats of military action against Nigeria, citing concerns over alleged killings of Christians.
- **[M23 Offensive in DR Congo](https://en.wikipedia.org/wiki/2025_Bukavu_offensive)**: The M23 rebel group launched a campaign in South Kivu, Democratic Republic of Congo, in February 2025, continuing the regional instability.
- **[Palestinian Authority Operation in Tubas](https://en.wikipedia.org/wiki/2024_Palestinian_Authority_operation_in_Tubas)**: The Palestinian Authority conducted "Operation Protecting the Nation" in the West Bank's Tubas Governorate from October to November 2024.
- **[Safadi Urges Gaza Ceasefire Adherence](https://jordantimes.com/news/local/safadi-urges-full-adherence-to-gaza-ceasefire-action-on-humanitarian-crisis)**: Jordan's Foreign Minister calls for complete compliance with the Gaza ceasefire and immediate humanitarian action.

#### Americas
- **[US Strikes on Drug-Running Boats](https://apnews.com/article/drug-cartels-hegseth-pacific-8f9f65dd67c0bc55b6dd70b109df0216)**: The US launched three strikes on alleged drug-trafficking vessels off Colombia's coast, resulting in 14 fatalities.
- **[Blue Jays Lead in World Series](https://www.cbc.ca/news/livestory/blue-jays-lead-3-1-in-fiery-winner-take-all-world-series-game-7-9.6963371)**: The Toronto Blue Jays are leading 3-1 in the decisive Game 7 of the World Series.
- **[Hurricane Melissa Devastation](https://www.breakingbelizenews.com/2025/11/01/international-news-hurricane-melissa-death-toll-rises-above-50-as-caribbean-reels-from-apocalyptic-damage/)**: Hurricane Melissa has caused over 50 deaths across the Caribbean, with Jamaica reporting 28 casualties and widespread "apocalyptic" damage.
- **[Catatumbo Clashes in Colombia](https://en.wikipedia.org/wiki/2025_Catatumbo_clashes)**: The National Liberation Army has conducted attacks in Colombia's Catatumbo region, escalating existing conflicts.

#### Asia
- **[Xi Returns from APEC Meeting](https://english.news.cn/20251101/f7cc64f12d5b4dab8e3dc753a89a208a/c.html)**: Chinese President Xi Jinping has returned to Beijing after attending the APEC meeting and conducting a state visit to South Korea.
- **[Kim Jong Un Inspects Hospital](https://www.nknews.org/2025/10/kim-jong-un-inspects-new-hospital-at-center-of-plans-for-health-care-reform/)**: North Korean leader Kim Jong Un visited a new hospital, highlighting healthcare reform plans amid ongoing economic challenges.
- **[Uzbekistan Qualifies for World Cup](https://uzreport.news/football/o-zbekiston-ilk-bor-mundialga-chiqdi)**: Uzbekistan has qualified for the FIFA World Cup for the first time in its history, marking a significant milestone for the country's football program.
- **[Ukrainian Drones Strike Russian Oil Terminal](https://kyivindependent.com/ukraine-reportedly-strikes-oil-terminal-in-russias-krasnodar-krai/)**: Ukrainian forces launched drone strikes against an oil terminal in Russia's Krasnodar Krai, continuing their strategy of targeting Russian infrastructure.

#### Europe
- **[UK Train Stabbing](https://www.smh.com.au/world/europe/multiple-people-stabbed-on-uk-train-in-appalling-incident-20251102-p5n733.html)**: Nine people suffered life-threatening injuries following a mass stabbing on a train in Cambridgeshire, UK, with two suspects arrested.
- **[German Nationals Killed in Avalanche](https://www.spiegel.de/panorama/suedtirol-fuenf-deutsche-bei-lawinenunglueck-getoetet-a-2f454053-926f-4c2f-bbbc-f28cea8b1220)**: Five German citizens have died in an avalanche accident in South Tyrol, Italy.
- **[Russia-Ukraine War Continues](https://www.rp.pl/swiat/art43271141-wojna-rosji-z-ukraina-dzien-1347)**: The conflict between Russia and Ukraine has reached day 1,347, with continued intense fighting in eastern regions.
- **[Estonian PM Criticizes Latvia](https://news.err.ee/1609845303/estonian-pm-criticizes-latvia-s-plan-to-withdraw-from-istanbul-convention)**: Estonia's Prime Minister has expressed disapproval of Latvia's decision to withdraw from the Istanbul Convention, which aims to combat violence against women.

#### Oceania
- **[Bus Fire in Auckland](https://www.stuff.co.nz/nz-news/360873281/bus-hits-overpass-and-catches-fire-aucklands-north-shore)**: A bus caught fire after hitting an overpass on Auckland's North Shore in New Zealand, with no injuries reported.
- **[UAE Aid to Hurricane-Hit Caribbean](https://www.thenationalnews.com/news/uae/2025/10/31/uae-sends-crucial-aid-to-caribbean-communities-hit-by-deadly-hurricane-melissa/)**: The United Arab Emirates has sent essential aid to Caribbean communities devastated by Hurricane Melissa.
- **[Tonga XIII Rugby Preparations](https://matangitonga.to/2025/10/30/tonga-xiii-set-take-kiwis-after-losing-samoa)**: Tonga's rugby team is preparing to face the New Zealand Kiwis after suffering a recent loss to Samoa.

---

## Detailed News Reports

### Political Developments

#### **[Trump Threatens Military Action Against Nigeria](https://guardian.ng/news/trump-threatens-military-action-in-nigeria-over-killing-of-christians/)**
Former US President Donald Trump has threatened military intervention in Nigeria over alleged killings of Christians. As reported by Nigeria's The Guardian and Austria's Kronen Zeitung, the controversial statement has raised significant concerns about potential US foreign policy shifts and implications for US-Nigerian relations. The threat has sparked widespread reactions regarding international intervention and religious freedom issues.

#### **[Palestinian Authority-West Bank Conflict](https://en.wikipedia.org/wiki/2024_Palestinian_Authority_operation_in_Tubas)**
The Palestinian Authority conducted an operation in the Tubas Governorate of the West Bank from October to late November 2024. The operation, dubbed "Protecting the Nation," was part of the broader Palestinian Authority–West Bank militias conflict and the ongoing Middle Eastern crisis. This development illustrates the complex internal Palestinian dynamics amid regional tensions.

#### **[Russia Reaffirms Support for Venezuela](http://www.eluniversal.com/internacional/219338/rusia-reafirma-su-solido-apoyo-a-venezuela-y-aboga-por-la-preservacion-de-america-latina-como-zona)**
Russia has reiterated its strong support for Venezuela and advocated for maintaining Latin America as a zone of peace, according to Venezuela's El Universal. This declaration comes amid heightened regional tensions and underscores Russia's strategic interests in the Americas, opposing external interference in Venezuelan affairs.

#### **[Estonia Criticizes Latvia's Convention Withdrawal](https://news.err.ee/1609845303/estonian-pm-criticizes-latvia-s-plan-to-withdraw-from-istanbul-convention)**
Estonia's Prime Minister has publicly criticized Latvia's decision to withdraw from the Istanbul Convention, which aims to combat violence against women. This disagreement highlights growing divisions within Baltic states on social policy issues and demonstrates Estonia's commitment to international frameworks protecting women's rights.

---

### Armed Conflicts & Security

#### **[M23 Campaign in DR Congo](https://en.wikipedia.org/wiki/2025_Bukavu_offensive)**
The M23 rebel group conducted an offensive in South Kivu, Democratic Republic of Congo, from February 5-16, 2025. This campaign represents a continuation of the ongoing M23 insurgency that began in 2022, further destabilizing the eastern DRC and threatening civilian populations in the region.

#### **[Catatumbo Clashes in Colombia](https://en.wikipedia.org/wiki/2025_Catatumbo_clashes)**
The National Liberation Army (ELN) has carried out attacks in Colombia's Catatumbo region, according to reports. These clashes occur in Norte de Santander and represent ongoing insurgent activity in Colombia despite peace efforts with other armed groups in recent years.

#### **[Russia-Ukraine War Day 1347](https://www.rp.pl/swiat/art43271141-wojna-rosji-z-ukraina-dzien-1347)**
The war between Russia and Ukraine has reached its 1,347th day, with continued intense fighting in eastern Ukraine. Polish newspaper Rzeczpospolita reports that Russia has intensified military operations while Ukraine's forces claim significant Russian casualties amid their counteroffensive efforts. Diplomatic talks remain stalled as both sides prepare for potential further escalation.

#### **[Ukrainian Drone Strikes on Russian Oil Terminal](https://kyivindependent.com/ukraine-reportedly-strikes-oil-terminal-in-russias-krasnodar-krai/)**
Ukrainian forces have conducted drone strikes against an oil terminal in Russia's Krasnodar Krai, The Kyiv Independent reports. This attack represents a continuation of Ukraine's strategy to disrupt Russia's logistics and fuel supply infrastructure, impacting Russia's military capabilities.

---

### Natural Disasters & Humanitarian Crises

#### **[Hurricane Melissa Death Toll Rises](https://www.breakingbelizenews.com/2025/11/01/international-news-hurricane-melissa-death-toll-rises-above-50-as-caribbean-reels-from-apocalyptic-damage/)**
Hurricane Melissa has caused over 50 deaths across the Caribbean, with survivors describing the aftermath as "apocalyptic," according to Breaking Belize News. In Jamaica alone, the death toll has reached 28, as reported by The Gleaner. Relief efforts are underway, but challenges persist due to the extensive destruction and ongoing weather concerns.

#### **[UAE Sends Aid to Caribbean Communities](https://www.thenationalnews.com/news/uae/2025/10/31/uae-sends-crucial-aid-to-caribbean-communities-hit-by-deadly-hurricane-melissa/)**
The United Arab Emirates has dispatched critical aid to Caribbean communities devastated by Hurricane Melissa, The National reports. This assistance aims to support recovery efforts and provide essential supplies for rebuilding and humanitarian relief in the affected areas.

#### **[November Rainfall Drenches Bangladesh](https://www.thedailystar.net/environment/weather/news/november-rain-drenches-dhaka-parts-country-4024631)**
Dhaka and various regions of Bangladesh are experiencing heavy rainfall in November, The Daily Star reports. The unseasonal downpour has led to disruptions in daily activities and raised concerns about potential flooding. Authorities are monitoring the situation closely to implement necessary safety measures.

#### **[Avalanche Kills Five Germans in South Tyrol](https://www.spiegel.de/panorama/suedtirol-fuenf-deutsche-bei-lawinenunglueck-getoetet-a-2f454053-926f-4c2f-bbbc-f28cea8b1220)**
Five German nationals died in an avalanche in South Tyrol, Italy, according to Der Spiegel. The incident occurred in a popular skiing area, and despite rescue operations, all five victims were confirmed deceased at the scene. This tragedy highlights the ongoing dangers of winter sports in Alpine regions.

---

### Crime & Justice

#### **[Louvre Heist Update](https://www.lemonde.fr/en/france/article/2025/11/01/louvre-heist-suspects-brought-before-paris-magistrates_6746992_7.html)**
Two additional suspects have been charged in connection with the Louvre museum heist in Paris, while three previously detained individuals have been released, Le Monde reports. The investigation into this high-profile art theft continues as authorities search for more leads and attempt to recover the stolen items.

#### **[Mass Stabbing on UK Train](https://www.smh.com.au/world/europe/multiple-people-stabbed-on-uk-train-in-appalling-incident-20251102-p5n733.html)**
Nine people suffered life-threatening injuries following a knife attack on a train in Cambridgeshire, UK. The Sydney Morning Herald reports that authorities have described the incident as "appalling," and emergency services responded promptly to the scene. Two individuals have been arrested in connection with the attack, according to multiple European news outlets.

#### **[Prosecutor Investigates Costa Rican OIJ Director](https://www.nacion.com/sucesos/fiscalia-indago-a-director-del-oij-randall-zuniga/7I7NITHXABGBFEIJRH3HHGNEVQ/story/)**
The Director of Costa Rica's Judicial Investigation Organization (OIJ), Randall Zúñiga, is under investigation for alleged sexual offenses, La Nación reports. This case has raised significant concerns about leadership within one of the country's most important law enforcement institutions and may have broader implications for Costa Rica's judicial system.

#### **[University Student Killed at Halloween Party in Bogotá](https://www.eltiempo.com/bogota/universidad-de-los-andes-se-pronucia-sobre-la-muetre-de-jaime-esteban-moreno-tras-una-fiesta-de-halloween-en-bogota-3505304)**
A student from the University of Los Andes was fatally shot during a Halloween party in Bogotá, Colombia, El Tiempo reports. The incident has raised concerns about safety during social events in the city, and authorities are investigating the circumstances surrounding this tragic event.

---

### Sports & Entertainment

#### **[Blue Jays in World Series Game 7](https://www.cbc.ca/news/livestory/blue-jays-lead-3-1-in-fiery-winner-take-all-world-series-game-7-9.6963371)**
The Toronto Blue Jays are leading 3-1 in the decisive Game 7 of the World Series, CBC News reports. This winner-take-all matchup has created a tense atmosphere as the Blue Jays seek their first World Series title in decades.

#### **[Springboks Dominate Japan at Wembley](https://www.news24.com/sport/rugby/springboks/live-test-rugby-springboks-v-japan-20251101-0530)**
South Africa's Springboks secured a convincing victory against Japan at Wembley Stadium, scoring nine tries in a dominant performance, according to News24. This win emphasizes the Springboks' strong position as one of the world's top rugby teams.

#### **[Sinner Dominates Zverev in Paris](https://www.corriere.it/sport/tennis/diretta-live/25_novembre_01/sinner-zverev-semifinale-parigi-diretta-risultato.shtml)**
Jannik Sinner delivered a commanding performance against Alexander Zverev, winning 6-0, 6-1 to advance to the final of the Paris tournament, Corriere della Sera reports. This victory highlights Sinner's exceptional form leading into the final matchup.

#### **[Uzbekistan Makes World Cup History](https://uzreport.news/football/o-zbekiston-ilk-bor-mundialga-chiqdi)**
Uzbekistan has qualified for the FIFA World Cup for the first time in its history, UzReport announces. This achievement marks a significant milestone for Uzbek football and has generated considerable national pride and excitement.

---

### Health & Science

#### **[Kim Jong Un Inspects New Hospital](https://www.nknews.org/2025/10/kim-jong-un-inspects-new-hospital-at-center-of-plans-for-health-care-reform/)**
North Korean leader Kim Jong Un has conducted an inspection of a newly built hospital as part of the country's healthcare reform plans, NK News reports. The facility aims to enhance North Korea's medical infrastructure and improve public health services, reflecting an ongoing focus on healthcare improvements amid broader economic challenges.

#### **[First X-ray Service for TORBA in Vanuatu](https://www.dailypost.vu/news/first-x-ray-service-for-torba/article_2e500028-0de0-51ac-87ea-95244a3dc33d.html)**
TORBA province in Vanuatu has received its first X-ray service, the Vanuatu Daily Post reports. This development significantly improves local healthcare access, enhancing diagnostic capabilities for residents in the region and reflecting ongoing efforts to upgrade medical facilities and services in remote areas.

#### **[Temple Stampede in India](https://nationnews.com/2025/11/01/indian-temple-stampede-kills-nine/)**
A stampede at an Indian temple has resulted in nine fatalities, according to Barbados' The Nation Newspaper. The incident occurred during a religious event that drew a large crowd. Local authorities are investigating the circumstances that led to this tragedy.

#### **[Gadgets Breaking Backs in Tajikistan](https://asiaplustj.info/ru/news/tajikistan/society/20251101/kak-gadzheti-lomayut-vashu-spinu)**
A report from Tajikistan's Asia-Plus highlights how increasing use of electronic gadgets is causing back problems among users. Poor posture while using smartphones and laptops is identified as a significant contributing factor to musculoskeletal issues. Experts recommend ergonomic adjustments and regular breaks to mitigate these health risks.

---

### Business & Economy

#### **[Public-Private Partnership in Comoros](https://alwatwan.net/economie/partenariat-public-priv%C3%A9-i-signature-d%E2%80%99une-convention-entre-le-minist%C3%A8re-des-finances-et-les-op%C3%A9rateurs-%C3%A9conomiques.html)**
Comoros' Ministry of Finance has signed a Public-Private Partnership agreement with economic operators, Al Watwan reports. This collaboration aims to enhance infrastructure development and economic growth, outlining the roles and responsibilities of both public and private entities involved.

#### **[ECCEA Criticism in Mauritius](https://lexpress.mu/s/les-grands-titres-de-lexpress-de-ce-dimanche-2-novembre-2025-551063)**
Roshnee Gunness, the wife of a minister, is at the center of criticism regarding the Early Childhood Care and Education Authority (ECCEA) in Mauritius, L'Express reports. This controversy raises questions about governance and potential conflicts of interest in the educational sector.

#### **[Major Romanian Bank Claims Right to Withdraw Funds](https://adevarul.ro/economie/o-mare-banca-din-romania-isi-da-dreptul-sa-retraga-2483554.html)**
A major Romanian bank has claimed the right to withdraw money from customers' accounts without consent under specific conditions, Adevărul reports. This policy has raised concerns about consumer rights and banking practices in Romania, highlighting tensions between financial institutions and their customers.

#### **[China Might Lift Sanctions on Hanwha Ocean](https://www.yna.co.kr/view/AKR20251102004800071?section=international/all)**
China is reportedly considering lifting sanctions on South Korea's Hanwha Ocean as part of a US-China trade agreement, according to South Korea's Yonhap News Agency. This potential development could signal improved economic relations between China and South Korea, occurring within the context of broader US-China trade negotiations.

---

### Cultural & Social News

#### **[Grand Egyptian Museum Showcases Pharaonic Civilization](https://www.annahar.com/culture/254192/%D8%A7%D9%84%D9%85%D8%AA%D8%AD%D9%81-%D8%A7%D9%84%D9%85%D8%B5%D8%B1%D9%8A-%D8%A7%D9%84%D9%83%D8%A8%D9%8A%D8%B1-%D8%A5%D8%AD%D9%8A%D8%A7-%D8%AD%D8%B6%D8%A7%D8%B1%D8%A9-%D8%A7%D9%84%D9%81%D8%B1%D8%A7%D8%B9%D9%86%D8%A9-%D8%A8%D8%B5%D9%88%D8%B1%D8%A9-%D8%B9%D8%B5%D8%B1%D9%8A%D8%A9)**
The Grand Egyptian Museum presents a stunning display of Pharaonic civilization in a contemporary context, Lebanon's An-Nahar reports. The museum features innovative exhibitions that showcase Egypt's rich archaeological heritage through modern design and interactive experiences. Sayyid Theyazin attended the inauguration representing Oman's Sultan, according to Times of Oman.

#### **[Lord of Miracles Final Procession](https://elcomercio.pe/lima/sucesos/senor-de-los-milagros-en-vivo-procesion-hoy-sexto-y-ultimo-recorrido-del-cristo-moreno-este-sabado-1-de-noviembre-de-2025-rutas-desvios-ultimas-noticias-lbposting-noticia/)**
The final procession of the year honoring the Lord of Miracles has taken place in Peru, El Comercio reports. This significant religious event drew thousands of devotees who expressed their faith through prayers and traditional practices, reinforcing cultural heritage and community solidarity.

#### **[Urn Forest in Izegem, Belgium](https://www.vrt.be/vrtnws/en/2025/10/28/urn-forest-in-izegem-allows-mourners-to-bury-the-deceased-in-nat/)**
Izegem, Belgium, has introduced an "urn forest" where mourners can bury the ashes of their loved ones in a natural setting, VRT NWS reports. This initiative provides an eco-friendly alternative to traditional burial methods, encouraging a deeper connection with nature during the grieving process.

#### **[Maputo Celebrates 138th Anniversary](https://www.jornalnoticias.co.mz/2025/11/01/rasaque-manhique-lanca-hoje-festividades-dos-138-anos-da-cidade-de-maputo/)**
Rasaque Manhique has officially launched celebrations for the 138th anniversary of Maputo, Mozambique's capital city, Noticias reports. The festivities aim to highlight the city's rich history and cultural heritage, with various events planned throughout the anniversary period.

---

### Technology & Innovation

#### **[Poseidon Ocean Crossing Capabilities](https://www.dnes.bg/a/2-svyat/698366-poseydon-mozhe-da-prekosi-okeana-bez-ogranicheniya-na-obhvata)**
The "Poseidon" vessel reportedly has the capability to navigate across oceans without range limitations, Bulgaria's dnes.bg reports. This technological advancement allows for extended maritime missions without the need for refueling or support, signifying a major breakthrough in naval engineering and operational efficiency.

#### **[Gen Z Employment Revolution](https://money.kompas.com/read/2025/11/02/073000926/revolusi-ketenagakerjaan-untuk-gen-z?source=headline)**
Generation Z is reshaping the employment landscape with preferences for flexible work arrangements and remote opportunities, Indonesia's Kompas reports. Employers are adapting to these demands by offering diverse benefits, including mental health support and career development. Technology and automation are influencing new job sectors, leading to a shift in required skills for entry-level positions.

#### **[Energy Company Imposes Electricity Restrictions in Kyrgyzstan](https://24.kg/obschestvo/349387_nesk_vvodit_ogranicheniya_moschnosti_elektroenergii_vsluchae_peregruzki_podstantsiy/)**
An energy company in Kyrgyzstan has implemented electricity capacity restrictions to prevent potential overloads at substations, 24.kg reports. The measure aims to ensure stability and reliability of the energy supply, though customers may experience limitations in electricity usage as part of this precautionary strategy.

#### **[Prime Minister Addresses Cyber Crime in Thailand](https://www.bangkokpost.com/thailand/general/3129814/pm-talks-tough-on-cyber-crime)**
Thailand's Prime Minister has emphasized the government's commitment to combating cyber crime, highlighting its increasing threat to national security, the Bangkok Post reports. New initiatives and stricter laws are being proposed to enhance cybersecurity measures and penalize offenders, with calls for greater collaboration between government agencies, private sector, and international partners.

---

As global events continue to unfold, this comprehensive summary provides insight into the key developments shaping our world today. From political tensions to technological advancements, natural disasters to cultural celebrations, the interconnected nature of our global society remains evident in the diverse headlines from around the globe.

In [33]:
prompt = f'''
Please translate this English newsletter into traditional Chinese.
English newsletter: {newsletter_en}
For names and locations, include both Chinese and English, for example:
English version: Trump Threatens Nigeria: Former US President Donald Trump has issued threats of military action against Nigeria, citing concerns over alleged killings of Christians.
Chinese version: 特朗普威脅尼日利亞：前美國總統唐納德·特朗普(Donald Trump)對尼日利亞(Nigeria)發出軍事行動威脅，理由是對基督徒被謀殺的指控表示關切。
Output only the traditional Chinese newsletter
'''

response = await client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": prompt}
    ]
)

In [34]:
response.choices[0].message.content

"全球新聞快訊 - 2025年11月2日  \n每日新聞直接來源於當地報紙  \n區域重點  \n非洲 / 中東  \n特朗普威脅尼日利亞：前美國總統唐納德·特朗普（Donald Trump）對尼日利亞（Nigeria）發出軍事行動威脅，理由是對基督徒被謀殺的指控表示關切。  \nM23在剛果民主共和國發起攻勢：M23叛軍於2025年2月在剛果民主共和國南基伍（South Kivu）發起攻擊，持續加劇區域不穩定。  \n巴勒斯坦權力機構在圖巴斯的行動：巴勒斯坦權力機構在約旦河西岸的圖巴斯省（Tubas Governorate）從2024年10月到11月進行了“保護國家行動”。  \n薩法迪呼籲遵守對加薩的停火協定：約旦外長呼籲完全遵守對加薩的停火協議並立即採取人道行動。  \n美洲  \n美國對毒品走私船隻發動空襲：美國在哥倫比亞（Colombia）海岸對涉嫌毒品走私的船隻發動了三次空襲，造成14人死亡。  \n藍鳥隊在世界大賽領先：多倫多藍鳥隊（Toronto Blue Jays）在世界大賽第七場決勝局中以3-1領先。  \n颶風梅利莎造成的毀滅：颶風梅利莎在加勒比地區造成超過50人死亡，牙買加（Jamaica）報告有28人遇難，並造成廣泛的“世界末日”損壞。  \n哥倫比亞卡塔廷波的衝突：國家解放軍在哥倫比亞的卡塔廷波（Catatumbo）地區發起攻擊，加劇了現有衝突。  \n亞洲  \n習近平從亞太經合組織會議回國：中國國家主席習近平在參加亞太經合組織會議後已返回北京。  \n金正恩視察醫院：北韓領導人金正恩（Kim Jong Un）參觀了一所新建醫院，強調在持續的經濟挑戰中推進醫療改革計劃。  \n烏茲別克斯坦晉級世界杯：烏茲別克斯坦首次晉級國際足聯世界杯，這對該國的足球計劃來說是一個重要的里程碑。  \n烏克蘭無人機攻擊俄國油港：烏克蘭軍隊對俄羅斯克拉斯諾達爾邊疆區的一個油港發動無人機攻擊，持續其針對俄國基礎設施的策略。  \n歐洲  \n英國列車 stabbing 事件：在英國劍橋郡的列車上發生集體刺傷事件，造成九人重傷，兩名嫌疑人被逮捕。  \n德國國籍人士在阿爾卑斯山雪崩中喪生：五名德國公民在意大利南蒂羅爾（South Tyrol）的一起雪崩事故中喪生。  \n俄烏戰爭持續：俄烏衝突已進入第1,347天，東部地區持續激烈戰鬥。  \n愛沙尼亞總理批評

In [ ]:
# websites = list(df[df["Link"].isna() & df["Headline"].notna()]["Website"])
# headlines = list(df[df["Link"].isna() & df["Headline"].notna()]["Headline"])
# combined = list(zip(websites, headlines))
# combined
websites = list(df[df["Headline"].isna()]["Website"])
websites

In [ ]:
# for website, headline in combined:
for website in websites:

    print(website)
    
    try:
        html_dict = extract_html.get_raw_html(website)
        html_string = html_dict["html"]
        clean_text = remove_html_tags(html_string)

        if len(html_string) > 200000:
            html_string_input = clean_text
        else:
            html_string_input = html_string

        prompt = f'''
        You are good at reading html text, visualize the content, and identify the headline of the day.
        Given this HTML text from a news website: {html_string_input}

        CRITICAL REQUIREMENTS:
        1. Identify the main headline of the day
        2. ALWAYS translate to English if the headline is in any other language
        3. Output ONLY the English headline text, nothing else
        4. Do not include any non-English words or phrases
        5. Ensure the headline is in proper English grammar and spelling
        6. If the headline contains any non-English characters or words, translate them to English

        IMPORTANT: The output must be 100% in English. If you cannot translate a headline to English, output "No headline found".
        '''

        response = await client.chat.completions.create(
#             model="anthropic/claude-haiku-4.5",
            model="openai/gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
        )

        headline = response.choices[0].message.content.strip()
        print(headline)

        soup = BeautifulSoup(html_string, "html.parser")

        # Extract all links
        links = {a.get_text(): a for a in soup.find_all('a')}

        # Get text without HTML tags
        # text = soup.get_text()
        links_on_page = ""

        # Append links to the text
        for link_text, url in links.items():
            links_on_page += f" [{link_text}]({url})"

        prompt = f'''
        Given this identified headline on a news website: "{headline}"
        Note that the headline is being translated into English if the original text is non-English.
        Please check if the link to that headline is in {links_on_page[:150000]}
        If so, please output only the link. 
        Note that sometime only the relative url is included. If so, you need to output the absolute url with the root website {website}.
        If no link is founded, just output 'N/A'
        Do not output anything else other than the link
        Do not output any html tags e.g., '<a href=', '</a>'
        '''

        response = await client.chat.completions.create(
            model="openai/gpt-4o-mini",
            messages=[
                {"role": "user", "content": prompt}
            ]
        )

        link = response.choices[0].message.content
        print(link)

        html_dict = extract_html.get_raw_html(link)
        html_string = html_dict["html"]
        clean_text = remove_html_tags(html_string)

        prompt = f'''
        You are a capable journalist. This is the headline of today: {clean_text}
        Please summarize in 2-3 English bullets to capture the key informaiton. Each bullet should be very concise.
        You may assume your readers know nothing about the country's news and politics
        The source may not be in English. But make sure the summary is in English
        '''

        response = await client.chat.completions.create(
            model="openai\gpt-4o-mini",
            messages=[
                {"role": "user", "content": prompt}
            ]
        )

        summary = response.choices[0].message.content
        print(summary)
        print("\n")
    
    except:
        continue

In [36]:
# Load a Markdown file
def load_markdown_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        content = file.read()
    return content

# Example usage
markdown_content = load_markdown_file('newsletter_20251103_103157.md')

In [37]:
display(Markdown(markdown_content))

# Headlines around the globe - Nov 03, 2025
Daily news sourced directly from local newspapers

## Regional Highlights

### Africa / Middle East
- **[MACC to Call Up Whistleblower Over Corruption Video](https://thesun.my/malaysia-news/macc-to-call-up-whistleblower-over-corruption-video-JN13789437)**: The Malaysian Anti-Corruption Commission will interview a whistleblower regarding a video linking a Sabah leader to corruption allegations. (The Sun, Malaysia)
- **[10 JAMB Staff Face Prosecution Over Corruption](https://guardian.ng/news/nigeria/national/10-jamb-staff-face-prosecution-over-corruption-allegations/)**: More than 10 staff members of Nigeria's Joint Admissions and Matriculation Board are being prosecuted for corruption allegations. (The Guardian, Nigeria)
- **[Houthi Forces Block Vital Goods into Taiz](http://www.yemenpost.net/Detail123456789.aspx?ID=3&SubID=8397&MainCat=3)**: Human Rights Watch reports that Houthi forces are obstructing essential supplies from reaching Taiz, worsening the humanitarian crisis in Yemen. (Yemen Post)
- **[President Isaias Attends Inauguration of Egypt's Grand Museum](https://shabait.com/2025/11/02/president-isaias-attends-inauguration-of-egypts-grand-museum/)**: Eritrea's President Isaias participated in the opening ceremony of Egypt's Grand Museum, highlighting cultural cooperation in the region. (Shabait)
- **[Xi Extends Congratulations on Grand Egyptian Museum Opening](https://english.news.cn/20251102/120c706554da4e2483be0adc5df4e1a4/c.html)**: Chinese President Xi Jinping congratulated Egyptian President el-Sisi on the opening of the Grand Egyptian Museum near the Giza Pyramids. (Xinhua)

### Americas
- **[Early Voting Ends With Over 735,000 Ballots Cast in NYC Mayor's Race](https://www.nytimes.com/live/2025/11/02/nyregion/nyc-mayor-election-news)**: New York City's mayoral election has seen significant turnout during early voting, indicating high public interest. (The New York Times, USA)
- **[Trump to Host U.S.-Central Asia Summit](http://qazinform.com/news/foreign-media-on-kazakhstan-trump-to-host-us-central-asia-summit-on-november-6-kazakhstan-discusses-exporting-uranium-to-finnish-nuclear-plants-aa8a29)**: Former President Trump is set to host a summit with Central Asian leaders on November 6, focused on strengthening regional ties. (Kazinform, Kazakhstan)
- **[Masked Individuals Storm Government Palace in Mexico](https://www.eluniversal.com.mx/estados/en-marcha-por-carlos-manzo-grupo-de-embozados-irrumpe-en-palacio-de-gobierno-en-morelia-lanzan-muebles-por-las-ventanas/)**: A group of protesters stormed the Government Palace in Mexico, throwing furniture from windows during unrest related to the assassination of a seventh mayor in Michoacán. (El Universal, Mexico)
- **[Dodgers Win Back-to-Back Championship, Yamamoto Earns World Series MVP](https://news.web.nhk/newsweb/na/na-k10014965451000)**: The Los Angeles Dodgers have secured consecutive World Series championships with Japanese pitcher Yoshinobu Yamamoto named MVP. (NHK, Japan)
- **[Trump Selects Susie Wiles as Chief of Staff](https://almomento.net/trump-elige-a-susie-wiles-jefa-gabinete-primera-mujer-en-cargo/)**: Donald Trump has appointed Susie Wiles, who previously managed his 2016 Florida campaign, as White House Chief of Staff. (ALMOMENTO.NET, Dominican Republic)

### Asia
- **[Indian Women Lift Historic Maiden ODI World Title](https://www.thehindu.com/sport/cricket/womens-odi-world-cup-cricket-india-defeat-south-africa-lift-maiden-world-cup/article70233864.ece)**: India's women's cricket team has won their first-ever ODI World Cup by defeating South Africa in the final. (The Hindu, India)
- **[Uzbekistan Qualifies for World Cup for First Time](https://uzreport.news/football/o-zbekiston-ilk-bor-mundialga-chiqdi)**: Uzbekistan has made history by qualifying for the FIFA World Cup for the first time. (UzReport, Uzbekistan)
- **[Russian Oil Tanker on Fire After Ukrainian Drone Strike](https://kyivindependent.com/ukraine-war-latest-russian-oil-tanker-on-fire-after-strike-by-ukrainian-drones-in-krasnodar-krai-source-says/)**: A Russian oil tanker is reportedly ablaze following a strike by Ukrainian drones in Krasnodar Krai. (Kyiv Independent, Ukraine)
- **[Kim Jong Un Inspects New Hospital](https://www.nknews.org/2025/10/kim-jong-un-inspects-new-hospital-at-center-of-plans-for-health-care-reform/)**: North Korean leader Kim Jong Un has visited a newly built hospital that's central to the country's healthcare reform plans. (NK News)
- **[6.3 Magnitude Earthquake Strikes Afghanistan](https://www.nrk.no/nyheter/afghanistan_-jordskjelv-med-6_3-i-styrke-1.17636653)**: A powerful earthquake has hit Afghanistan, resulting in at least seven fatalities with rescue efforts underway. (NRK, Norway)

### Europe
- **[Protests Erupt Outside Serbia's Parliament](https://www.panorama.com.al/trazira-perpara-kuvendit-te-serbise-protestuesit-hedhin-flakadane-dhe-molotove-reagon-vucic-u-bej-thirrje-per-paqe/)**: Demonstrators threw torches and Molotov cocktails outside Serbia's parliament, with President Vučić calling for peace amid the unrest. (Gazeta Panorama, Albania)
- **[New Violence in Serbia – Vučić Promises New Elections](https://www.svt.se/nyheter/utrikes/nya-valdsamheter-i-serbien-vucic-lovar-nyval)**: President Aleksandar Vučić has announced plans for new elections in response to renewed violence across Serbia. (SVT, Sweden)
- **[37 People Arrested After Riot Near Serbian Assembly](https://www.blic.rs/vesti/politika/oglasio-se-mup-uhapseno-37-lica-nakon-nereda-kod-skupstine-srbije/689609c)**: The Serbian Interior Ministry announced that 37 individuals were arrested for disturbing public order during riots near the assembly. (Blic, Serbia)
- **[NATO Task Force Clears Historical Sea Mines in Gulf of Finland](https://yle.fi/a/74-20191775)**: NATO forces have successfully cleared historical sea mines in the Gulf of Finland, enhancing maritime safety in the region. (YLE, Finland)
- **[Arrest After Man Dies Following Dublin Assault](https://www.rte.ie/news/2025/1102/1541720-tyrellstown-dublin/)**: Irish authorities have made an arrest after a man in his 20s died following an assault in Dublin. (RTÉ News, Ireland)

### Oceania
- **[Ley's Leadership in Peril as Senior Liberals Move to Dump Net Zero](https://www.smh.com.au/politics/federal/ley-s-top-lieutenants-lean-towards-scrapping-net-zero-as-leadership-in-peril-20251103-p5n78b.html)**: Senior Liberal Party members are pushing to abandon the net zero emissions policy, threatening the leadership position of the party leader. (The Sydney Morning Herald, Australia)
- **[First X-ray Service for TORBA](https://www.dailypost.vu/news/first-x-ray-service-for-torba/article_2e500028-0de0-51ac-87ea-95244a3dc33d.html)**: The TORBA region in Vanuatu has launched its first-ever X-ray service, significantly improving local healthcare capabilities. (Vanuatu Daily Post)
- **[Floats Parade Brings Festivity to Nuku'alofa](https://matangitonga.to/2025/11/03/floats-parade-brings-festivity-nukualofa)**: A vibrant parade in Nuku'alofa, Tonga, showcased elaborate displays and brought together local communities in celebration. (Matangi Tonga)
- **[Deacon's Primary to Reopen After Upgrades](https://nationnews.com/2025/11/02/deacon-primary-to-reopen-monday-following-upgrade-works/)**: Deacon's Primary School in Barbados will reopen on Monday following the completion of facility improvements. (The Nation Newspaper)
- **[Halt the Parades for Now, Says Police](https://www.samoaobserver.ws/category/samoa/116728)**: Samoan police have announced a temporary suspension of parades due to safety concerns. (Samoa Observer)

## Detailed News Coverage

### Political Developments

**[Israel Receives Remains of Three Hostages from Gaza](https://www.lemonde.fr/en/international/article/2025/11/02/israel-receives-the-remains-of-three-hostages-from-gaza-as-fragile-ceasefire-holds_6747028_4.html)**: Israel has received the remains of three hostages from Gaza amid a fragile ceasefire. According to former President Trump, one of the bodies released is that of American-Israeli IDF soldier Omer Neutra. The transfer occurs as part of ongoing negotiations between the parties, with the ceasefire remaining tenuous. (Le Monde, France)

**[Trump: Putin Is a Strong Leader Whom One Cannot Play With](https://ria.ru)**: Former U.S. President Trump has characterized Russian President Putin as a strong leader who cannot be manipulated, signaling potential shifts in diplomatic approaches should Trump return to office. (RIA Novosti, Russia)

**[Paul Biya Re-elected in Cameroon Presidential Election](https://www.cameroon-tribune.cm/article.html/73520/fr.html/presidentielle-2025-paul-biya-reelu)**: Paul Biya has secured another term as President of Cameroon following the 2025 presidential election, extending his decades-long rule over the country. (Cameroon Tribune)

**[Nigerian Urges Trump Meeting After Military Action Threat](https://guardian.ng/news/nigeria/national/nigeria-urges-trump-meeting-after-military-action-threat/)**: Nigeria is seeking diplomatic engagement with former President Trump following his threat of military action regarding regional issues, highlighting tensions in U.S.-Nigeria relations. (The Guardian Nigeria)

**[Former MP Selmon Walters to Be Laid to Rest Today](https://www.searchlight.vc/breaking-news/2025/11/01/former-mp-selmon-walters-laid-rest-today/)**: The funeral for former Member of Parliament Selmon Walters is taking place today in Saint Vincent and the Grenadines, drawing political colleagues and community members. (Searchlight)

### Conflicts and Tensions

**[Palestinians Face Hunger, Cold and Loss Amid Ongoing Israeli Siege](https://www.aljazeera.com/news/2025/11/2/palestinians-face-hunger-cold-and-loss-amid-ongoing-israeli-siege-on-gaza)**: Gaza residents continue to experience severe shortages of food and essential supplies due to the ongoing Israeli siege, with winter conditions exacerbating the humanitarian crisis. (Al Jazeera, Qatar)

**[Gaza Children Back to School](https://kuwaittimes.com/article/35248/top-stories/gaza-children-back-to-school/)**: Children in Gaza are returning to classrooms after disruptions from recent conflicts, with schools implementing safety measures and mental health support to restore educational normalcy. (Kuwait Times)

**[Israeli Media: Israel Is Preparing to Open a New Front in Iraq](https://www.entekhab.ir/fa/news/893035/رسانه‌-عبری-اسرائیل-در-تدارک-گشودن-جبهه‌ای-جدید-در-عراق-است)**: Israeli media reports suggest that Israel is planning military operations in Iraq, potentially escalating regional tensions in the Middle East. (Entekhab, Iran)

**[Russian Forces Bombard Kherson Region, War in Ukraine Day 1348](https://www.zdg.md/stiri/live-text-fortele-ruse-bombardeaza-regiunea-herson-mai-multe-persoane-sunt-ranite-razboi-in-ucraina-ziua-1348/)**: Russian forces continue to target Ukraine's Kherson region, resulting in multiple injuries as the conflict enters its 1,348th day. (Ziarul de Gardă, Republic of Moldova)

**[Turkey's Foreign Minister: Ankara Expects PKK Activities in Iraq to Cease](https://www.dnes.bg/a/2-svyat/698439-turskiyat-vanshen-ministar-ankara-ochakva-deynostite-na-pkk-v-irak-da-badat-prekrateni)**: Turkey's Foreign Minister has stated that Ankara anticipates an end to PKK activities in Iraq as part of ongoing military and diplomatic efforts in the region. (dnes.bg, Bulgaria)

### Economic News

**[Credit Cards in Costa Rica: One Bank Controls Nearly Half of Business](https://www.nacion.com/economia/tarjetas-de-credito-en-costa-rica-un-banco/GRGYTDUJTBHN7DWOZGLFM6XHGU/story/)**: A single bank dominates Costa Rica's credit card market with nearly 50% market share, raising concerns about competition and consumer choice in the financial sector. (La Nación, Costa Rica)

**[Inflation Falls to 2% in October in Belgium](https://www.vrt.be/vrtnws/en/2025/10/30/inflation-falls-to-2-per-cent-in-october-what-became-cheaper-w/)**: Belgium's inflation rate decreased to 2% in October, with lower prices for food and energy offset by rising costs in housing and transportation, signaling a mixed economic outlook. (VRT NWS, Belgium)

**[GIPF Faces N$1 Billion Loss After Underperforming Investments](https://www.namibian.com.na/gipf-faces-n1-billion-loss-after-underperforming-offshore-investments-in-sa/)**: Namibia's Government Institutions Pension Fund is projecting losses of N$1 billion due to poor performance of its South African offshore investments. (The Namibian)

**[The Municipalities Submit 550 Licenses to Urban Planning](https://www.bondia.ad/societat/els-comuns-entren-550-llicencies-a-urbanisme-71-mes-que-el-2023)**: Andorra's municipalities have submitted 550 licenses to Urban Planning, representing an increase of 71 licenses compared to 2023 and indicating growth in urban development. (Bondia.ad)

**[Ethiopia's Economic Growth Creates Opportunities for Manufacturing](https://www.ena.et/web/eng/w/eng_7634853)**: Ethiopia's Ministry highlights strong economic growth that's opening significant opportunities in the manufacturing sector, aiming to enhance industrialization and attract investment. (Ethiopian News Agency)

### Sports News

**[Coquimbo Unido Becomes First Champion of Chilean Football](https://www.emol.com/noticias/Deportes/2025/11/02/1182103/coquimbo-unido-campeon-futbol-chileno.html)**: Coquimbo Unido has made history by becoming the first champion of Chilean football, marking a significant milestone for the club. (emol.com, Chile)

**[Sinner Beats Auger-Aliassime in Tennis Final](https://www.corriere.it/sport/tennis/diretta-live/25_novembre_02/sinner-auger-aliassime-finale-parigi-diretta.shtml)**: Jannik Sinner defeated Felix Auger-Aliassime in a thrilling tennis match, with his powerful backhand down the line proving decisive. (Corriere della Sera, Italy)

**[Football Ligue 1: Lupopo Defeats Mazembe, Vita Club Held by Maniema](https://radiookapi.net/2025/11/02/actualite/sport/football-ligue-1-lupopo-renverse-mazembe-vita-club-freine-par-maniema)**: In DR Congo's Ligue 1, Lupopo secured a comeback victory against Mazembe, while Vita Club was held to a draw by Maniema Union. (Radio Okapi, Democratic Republic of the Congo)

**[Oklahoma Remains NBA Benchmark, Hartenstein Shines](https://www.spiegel.de/sport/basketball/nba-oklahoma-city-bleibt-ungeschlagen-und-isaiah-hartenstein-brilliert-a-957ba71d-ff31-4dba-b3da-d62bec074000)**: The Oklahoma City Thunder continues its strong start to the NBA season, while German player Isaiah Hartenstein delivers impressive performances. (Der Spiegel, Germany)

**[Germany U-17 Reaches World Cup Final After Victory Over Qatar](https://www.gabonews.ga/sports/handball/39/lallemagne-u-17-en-finale-mondiale-apres-une-victoire-ecrasante-sur-le-qatar/)**: Germany's Under-17 team has secured a place in the World Cup final following a dominant win against Qatar. (Gabonews, Gabon)

### Health & Science

**[Justice Sector Commits to Digital Case Management in Bhutan](https://www.bbs.bt/235112/)**: Key institutions in Bhutan's justice sector are implementing new digital case management systems to enhance efficiency and accessibility within judicial processes. (Bhutan Broadcasting Service)

**[Unseasonal Rains Devastate Paddy Harvest Across Nepal](https://kathmandupost.com/national/2025/11/02/unseasonal-rains-devastate-paddy-harvest-across-nepal)**: Unexpected rainfall in Nepal has severely damaged paddy harvests, threatening food security as farmers report significant crop losses. (The Kathmandu Post)

**[Jamaica Braces for Food Shortages After Hurricane Melissa](https://www.breakingbelizenews.com/2025/11/02/jamaica-braces-for-egg-and-food-shortages-after-hurricane-melissa/)**: Jamaica is preparing for potential egg and food shortages following the impact of Hurricane Melissa, which has disrupted supply chains and agricultural production. (Breaking Belize News)

**[Menopause: A Silent and Transformative Journey](https://www.anacao.cv/noticia/2025/11/03/menopausa-uma-travessia-silenciosa-e-transformadora/)**: A feature in Cabo Verde's A Nação newspaper highlights the often-overlooked impacts of menopause on women's physical and emotional health, emphasizing the need for greater awareness and support. (A Nação)

**[Medics Suspend Strike at Central Hospital of Nampula](https://www.jornalnoticias.co.mz/2025/11/02/medicos-decidem-suspender-greve-no-hospital-central-de-nampula/)**: Medical staff at Mozambique's Central Hospital of Nampula have decided to suspend their strike action following negotiations addressing their concerns. (Noticias)

### Society & Culture

**[From Hardships to Hope: Daikundi Woman Leads Change](https://pajhwok.com/2025/11/02/from-hardships-to-hope-daikundi-woman-leads-change/)**: A determined woman from Afghanistan's Daikundi province is transforming her community despite significant challenges, leading initiatives to improve living conditions. (Pajhwok Afghan News)

**[That Luang Festival Kicks Off in Laos](https://vientianetimes.org.la/freefreenews/freecontent_212_That_y25.php)**: The That Luang festival has begun in Laos, celebrating cultural heritage and religious traditions while uniting participants from diverse backgrounds. (Vientiane Times)

**[Blood Feud in Vorizia: Scenes of Ancient Tragedy](https://www.kathimerini.gr/society/563899600/vorizia-skines-archaias-tragodias-meta-to-foniko-se-ypsisti-epifylaki-i-el-as/)**: A murder in Vorizia, Greece, has triggered a violent blood feud reminiscent of ancient tragedies, with police on high alert to prevent further violence. (Kathimerini, Greece)

**[A Man Earns More – a Norm. A Woman Earns More – a Problem?](https://www.delfi.lv/life/56017194/attiecibas/120093401/virietis-pelna-vairak-norma-sieviete-pelna-vairak-problema)**: A Latvian article examines societal double standards where men's higher earnings are accepted while women's financial success often faces scrutiny and criticism. (Delfi, Latvia)

**[Why an Italian City Named Streets After Luxembourg Cities](https://www.wort.lu/panorama/warum-eine-stadt-in-italien-strassen-nach-luxemburger-staedten-benannt-hat/101839410.html)**: A city in Italy has dedicated street names to Luxembourg cities to strengthen cultural ties and international friendship. (Luxemburger Wort, Luxembourg)

### Crime & Justice

**[MACC to Call Up Whistleblower Over Corruption Video](https://thesun.my/malaysia-news/macc-to-call-up-whistleblower-over-corruption-video-JN13789437)**: The Malaysian Anti-Corruption Commission (MACC) will interview a whistleblower regarding a video linking a Sabah leader to corruption allegations. Chief Commissioner Tan Sri Azam Baki stated the investigation would proceed based on evidence from the whistleblower's statement. (The Sun, Malaysia)

**[Vehicle Licence Bribery: MACC Detains Officers](https://thesun.my/malaysia-news/vehicle-licence-bribery-macc-detains-officers-ex-staff-of-govt-agency-GH13507221)**: Two staff members and a former employee of a Malaysian ministry's agency have been detained by the MACC for alleged involvement in corrupt practices related to vehicle licensing. (The Sun, Malaysia)

**[Bushiri Secures Victory Against Extradition](https://malawi24.com/2025/10/31/bushiri-secures-victory-against-extradition-high-court-declares-it-unlawful-and-unconstitutional/)**: Malawi's High Court has ruled that the extradition of Shepherd Bushiri is unlawful and unconstitutional, securing a legal victory for Bushiri who faces charges in South Africa. (Malawi24)

**[Police in Big Drug Bust at Foulis, Guyana](https://www.stabroeknews.com/2025/11/02/news/guyana/police-in-big-drug-bust-at-foulis/)**: Guyanese authorities have conducted a significant drug bust in Foulis, making multiple arrests as part of a broader crackdown on drug-related activities. (Stabroek News)

**[Thieves Stop Trains by Stealing Cables](https://www.novinky.cz/clanek/domaci-kvuli-kradezi-kabelu-stoji-na-morave-vlaky-40546761)**: In the Czech Republic, thieves have disrupted train services by stealing essential cables, causing significant delays and operational challenges. (Novinky.cz)

## Looking Forward

As November begins, several key developments are worth monitoring in the coming days:

1. The fragile ceasefire between Israel and Hamas, especially following the return of hostage remains
2. Political instability in Serbia, with President Vučić promising new elections amid violent protests
3. Trump's planned U.S.-Central Asia summit on November 6 and the implications for regional diplomacy
4. Recovery efforts in regions affected by natural disasters, including Afghanistan (earthquake) and Jamaica (Hurricane Melissa)
5. The emerging political situation in Mexico following a string of assassinations of local officials

The global landscape continues to evolve with significant implications for international relations, economic stability, and humanitarian concerns across multiple regions.

In [40]:
import markdown
html_content = markdown.markdown(markdown_content)

In [41]:
html_content

'<h1>Headlines around the globe - Nov 03, 2025</h1>\n<p>Daily news sourced directly from local newspapers</p>\n<h2>Regional Highlights</h2>\n<h3>Africa / Middle East</h3>\n<ul>\n<li><strong><a href="https://thesun.my/malaysia-news/macc-to-call-up-whistleblower-over-corruption-video-JN13789437">MACC to Call Up Whistleblower Over Corruption Video</a></strong>: The Malaysian Anti-Corruption Commission will interview a whistleblower regarding a video linking a Sabah leader to corruption allegations. (The Sun, Malaysia)</li>\n<li><strong><a href="https://guardian.ng/news/nigeria/national/10-jamb-staff-face-prosecution-over-corruption-allegations/">10 JAMB Staff Face Prosecution Over Corruption</a></strong>: More than 10 staff members of Nigeria\'s Joint Admissions and Matriculation Board are being prosecuted for corruption allegations. (The Guardian, Nigeria)</li>\n<li><strong><a href="http://www.yemenpost.net/Detail123456789.aspx?ID=3&amp;SubID=8397&amp;MainCat=3">Houthi Forces Block Vital

In [6]:
website = "https://asia.nikkei.com/business/tech/semiconductors"

In [8]:
html_dict = extract_html.get_raw_html(website)
html_string = html_dict["html"]
clean_text = remove_html_tags(html_string)

if len(html_string) > 200000:
    html_string_input = clean_text
else:
    html_string_input = html_string

In [9]:
html_string

'<!DOCTYPE html>\n<html lang="en">\n <head>\n  <meta charset="utf-8"/>\n  <link href="/images/frontend/favicons/default.png" rel="icon" type="image/png"/>\n  <link href="/images/frontend/favicons/144x144.png" rel="icon" sizes="144x144" type="image/png"/>\n  <link href="/images/frontend/favicons/288x288.png" rel="icon" sizes="288x288" type="image/png"/>\n  <link href="/images/frontend/favicons/144x144.png" rel="apple-touch-icon-precomposed" sizes="144x144"/>\n  <link href="/images/frontend/favicons/288x288.png" rel="apple-touch-icon-precomposed" sizes="288x288"/>\n  <meta content="IE=Edge" http-equiv="X-UA-Compatible"/>\n  <meta content="yes" name="mobile-web-app-capable"/>\n  <meta content="yes" name="apple-mobile-web-app-capable"/>\n  <meta content="Nikkei Asia" name="application-name"/>\n  <meta content="Nikkei Asia" name="apple-mobile-web-app-title"/>\n  <meta content="#0076bf" name="theme-color"/>\n  <meta content="#0076bf" name="msapplication-navbutton-color"/>\n  <meta content="b

In [10]:
from pydantic import BaseModel, Field
from pydantic import ValidationError
from typing import List

In [11]:
class Headlines(BaseModel):
    headlines: List[str]

In [12]:
prompt = f"""
You are a web scraper that extracts news headlines from a news site's homepage HTML.

Here is the full HTML of the news site:

```{html_string_input}```

Extract the top 10 most prominent headlines of the day. 
Focus on the main stories shown at the top of the page (usually the hero story and the next 9 featured stories). 
Ignore navigation links, video titles, weather, sport sub-sections, and advertisements.

Return ONLY a JSON object that matches this exact schema:
{{
  "headlines": [
    "headline 1",
    "headline 2",
    ...
  ]
}}
Do not include any explanations, markdown, or extra text. 
If there are fewer than 10 prominent headlines, return all of them.
"""

In [13]:
response = await client.chat.completions.create(
    model="openai/gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
    temperature=0.2,        # low temperature for reliable extraction
#     max_tokens=1500,
)

In [14]:
raw_content = response.choices[0].message.content.strip()

In [15]:
raw_content

'{\n  "headlines": [\n    "TSMC to make advanced chips for AI at 2nd Japan plant",\n    "Chip titan will switch to 3-nm node, helping Japan secure stable supply of powerful products",\n    "HP, Dell, Acer and Asus mull using Chinese memory chips amid supply crunch",\n    "Japan chipmaker Rapidus tops $1bn in private investment as IBM aims to join",\n    "AMD rakes in $390m from AI chip sales to China but warns of uncertainty",\n    "AI effect starts to flip \'Korea discount\' on stocks into a premium",\n    "Apple supplier Nittobo to roll out improved glass cloth crucial to AI chips",\n    "What bubble? Big Tech\'s AI spending spree gathers pace in 2026",\n    "Japan\'s Renesas to sell timing solution segment to US-based SiTime for $3bn",\n    "TSMC\'s American expansion is not a surrender -- it\'s insurance for Taiwan"\n  ]\n}'

In [16]:
try:
    # Parse the JSON output directly into your Pydantic model
    data = json.loads(raw_content)
    top_headlines = Headlines(**data)
    
    # Now you have a clean list of strings
    print("Top 10 Headlines:")
    for i, headline in enumerate(top_headlines.headlines, 1):
        print(f"{i}. {headline}")
        
except json.JSONDecodeError as e:
    print("LLM did not return valid JSON:", e)
    print("Raw output:", raw_content)
except ValidationError as e:
    print("Validation error:", e)
    print("Raw output:", raw_content)

Top 10 Headlines:
1. TSMC to make advanced chips for AI at 2nd Japan plant
2. Chip titan will switch to 3-nm node, helping Japan secure stable supply of powerful products
3. HP, Dell, Acer and Asus mull using Chinese memory chips amid supply crunch
4. Japan chipmaker Rapidus tops $1bn in private investment as IBM aims to join
5. AMD rakes in $390m from AI chip sales to China but warns of uncertainty
6. AI effect starts to flip 'Korea discount' on stocks into a premium
7. Apple supplier Nittobo to roll out improved glass cloth crucial to AI chips
8. What bubble? Big Tech's AI spending spree gathers pace in 2026
9. Japan's Renesas to sell timing solution segment to US-based SiTime for $3bn
10. TSMC's American expansion is not a surrender -- it's insurance for Taiwan


In [17]:
for headline in top_headlines.headlines:
    print(headline)

TSMC to make advanced chips for AI at 2nd Japan plant
Chip titan will switch to 3-nm node, helping Japan secure stable supply of powerful products
HP, Dell, Acer and Asus mull using Chinese memory chips amid supply crunch
Japan chipmaker Rapidus tops $1bn in private investment as IBM aims to join
AMD rakes in $390m from AI chip sales to China but warns of uncertainty
AI effect starts to flip 'Korea discount' on stocks into a premium
Apple supplier Nittobo to roll out improved glass cloth crucial to AI chips
What bubble? Big Tech's AI spending spree gathers pace in 2026
Japan's Renesas to sell timing solution segment to US-based SiTime for $3bn
TSMC's American expansion is not a surrender -- it's insurance for Taiwan


In [18]:
soup = BeautifulSoup(html_string, "html.parser")

# Extract all links
links = {a.get_text(): a for a in soup.find_all('a')}

# Get text without HTML tags
# text = soup.get_text()
links_on_page = ""

# Append links to the text
for link_text, url in links.items():
    links_on_page += f" [{link_text}]({url})"

In [19]:
len(links_on_page)

71307

In [20]:
headline = top_headlines.headlines[0]
headline

'TSMC to make advanced chips for AI at 2nd Japan plant'

In [21]:
prompt = f'''
Given this identified headline on a news website: "{headline}"
Note that the headline is being translated into English if the original text is non-English.
Please check if the link to that headline is in {links_on_page[:150000]}
If so, please output only the link. 
Note that sometime only the relative url is included. If so, you need to output the absolute url with the root website {website}.
If no link is founded, just output 'N/A'
Do not output anything else other than the link
Do not output any html tags e.g., '<a href=', '</a>'
'''

response = await client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": prompt}
    ]
)

In [22]:
response.choices[0].message.content.strip()

'https://asia.nikkei.com/business/tech/semiconductors/tsmc-to-make-advanced-chips-for-ai-at-2nd-japan-plant'

In [5]:
def scrape_data(url):
    options = webdriver.ChromeOptions()
    options.add_argument('--headless')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    # These two lines help avoid detection and reduce cookie banner issues
    options.add_argument('--disable-blink-features=AutomationControlled')
    options.add_experimental_option("excludeSwitches", ["enable-automation"])

    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    wait = WebDriverWait(driver, 20)

    try:
        driver.get(url)

        # === STEP 1: Kill the cookie banner first ===
        cookie_accepted = False
        for _ in range(3):  # retry a few times
            try:
                # Most common OneTrust "Accept All" buttons (try several variants)
                accept_button = None
                accept_selectors = [
                    "#onetrust-accept-btn-handler",           # Most common
                    ".onetrust-close-btn-handler",            # Sometimes there's a close X
                    "#onetrust-button-group #accept-recommended-btn-handler",
                    "button[aria-label*='Accept' i]",         # Case-insensitive "Accept"
                    "button:contains('Accept')",              # fallback (rare)
                    "#onetrust-pc-btn-handler",               # "Reject All" or "Manage" sometimes works too
                ]

                for selector in accept_selectors:
                    try:
                        accept_button = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, selector)))
                        driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", accept_button)
                        driver.execute_script("arguments[0].click();", accept_button)  # JS click = more reliable
                        print("Cookie banner accepted")
                        cookie_accepted = True
                        time.sleep(2)
                        break
                    except:
                        continue
                
                if cookie_accepted:
                    break
            except:
                pass
            time.sleep(1)

        # === STEP 2: Now safely click "7 DAYS" ===
        try:
            seven_days = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "label[for='Daterange2']")))
            driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", seven_days)
            driver.execute_script("arguments[0].click();", seven_days)
            print("'7 DAYS' clicked successfully")
            time.sleep(2)  # let the table reload
        except Exception as e:
            print("Could not click 7 DAYS:", e)

        # === STEP 3: Wait for table content to actually change/load ===
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "table tr td")))
        time.sleep(3)  # small buffer – you can make this smarter later

        # === Your original scraping logic here ===
        extracted_data = []
        table_rows = driver.find_elements(By.CSS_SELECTOR, 'table tr')
        
        for row in table_rows:
            cells = row.find_elements(By.TAG_NAME, 'td')
            if cells and len(cells) >= 4:
                date_time_text = cells[0].text.strip()
                code_text = cells[1].text.strip()
                company_text = cells[2].text.strip()
                headline_text = cells[3].text.strip()
                
                # Find PDF link
                link = ""
                all_links = cells[3].find_elements(By.TAG_NAME, 'a')
                for link_element in all_links:
                    href = link_element.get_attribute('href')
                    if href and '.pdf' in href.lower():
                        link = href
                        break
                
                # Parse date/time
                date_str = date_time_text
                time_str = ""
                if ' ' in date_time_text:
                    date_str, time_str = date_time_text.split(' ', 1)
                
                # Handle multi-company rows
                if '\n' in code_text and '\n' in company_text:
                    codes = [c.strip() for c in code_text.split('\n')]
                    companies = [c.strip() for c in company_text.split('\n')]
                    
                    for code, company in zip(codes, companies):
                        row_data = process_row_data(date_str, time_str, code, company, headline_text, link)
                        extracted_data.append(row_data)
                else:
                    row_data = process_row_data(date_str, time_str, code_text, company_text, headline_text, link)
                    extracted_data.append(row_data)
                    
        return extracted_data
    
    except Exception as e:
        print(f"Error during scraping: {e}")
        return []

    finally:
        driver.quit()


# Helper function to avoid code duplication
def process_row_data(date_str, time_str, code, company, headline_text, link):
    row_data = {
        "date": date_str,
        "time": time_str,
        "code": code,
        "company": company,
        "category": "",
        "title": "",
        "link": link,
        "summary": ""
    }
    
    if '\n' in headline_text:
        parts = headline_text.split('\n', 1)
        if len(parts) == 2:
            category, title = parts
            row_data["category"] = category.strip()
            title = title.split('...More')[0].strip()
            row_data["title"] = title
    else:
        row_data["title"] = headline_text.strip()
    
    return row_data

In [6]:
def prescreen(title):
    if 'Next Day Disclosure Return' in title:
        return False
    else:
        return True

In [7]:
async def is_relevant(title):
    prompt = f"""
    You are a research analyst who is good at screening out minor and irrelevant news of a company by reading the title of an announcement.
    In general, regular returns (e.g., Next Day Disclosure Returns), proxy statement, amendments to articles are not relevant.
    Given the title of the announcement: {title}, please output only 'relevant' or 'irrelevant'
    """
    
    response = await client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    
    if response.choices[0].message.content.strip() == "irrelevant":
        return "irrelevant"
    else:
        return "relevant"

In [8]:
def generate_summary(url):
    with tempfile.TemporaryDirectory() as temp_dir:
        pdf_path = os.path.join(os.getcwd(), "document.pdf")
        response = requests.get(url, stream=True, timeout=60)
        response.raise_for_status()
        with open(pdf_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)

    documents = SimpleDirectoryReader(
        input_files=[pdf_path],
    ).load_data()                
            
    summary_index = SummaryIndex(documents)
    
    summary_query_engine = summary_index.as_query_engine(
        response_mode="tree_summarize",
        use_async=True,
    )    

    prompt='''
    Please summarize the document in 4-8 bullets in a concise manner. Avoid legal jargon.
    '''
    
    response = summary_query_engine.query(prompt)
    
    return response.response

In [9]:
data = scrape_data("https://www1.hkexnews.hk/listedco/listconews/index/lci.html?lang=en")

Cookie banner accepted
'7 DAYS' clicked successfully


In [10]:
len(data)

1023

In [11]:
stock_list = [
    "09988", #Alibaba
    "00700", #Tencent
    "09888", #Baidu
    "09618", #JD
    "09999", #Netease
    "01024", #Kuaishou
    "03690", #Meituan
    "01810", #Xiaomi
    "01211", #BYD
    "09866", #NIO
    "09868", #Xpeng
    "02015", #Li Auto
    "09863", #Leap Motor
    "00020", #Sensetime
    "09660", #Horizon Robotics
    "02525", #Hesai
    "02665", #Seyond
    "02498", #Robosense
    "02590", #Geekplus
    "02432", #Dobot
    "09880", #Ubtech
    "02026", #pony
    "00800", #weride
]

In [12]:
len(data)

1023

In [13]:
from datetime import datetime, timedelta

# Get today's date
today = datetime.now()

# Find the previous weekday
if today.weekday() == 0:  # Monday
    previous_weekday = today - timedelta(days=3)  # Go back to Friday
else:
    previous_weekday = today - timedelta(days=1)  # Go back one day

# Format the dates
formatted_today = today.strftime('%d/%m/%Y')
formatted_previous_weekday = previous_weekday.strftime('%d/%m/%Y')

# Output the result
data_range = [formatted_previous_weekday, formatted_today]
print(data_range)

['16/12/2025', '17/12/2025']


In [14]:
relevant_target_data_ytd = [row for row in data if 
                   row['date'] in data_range and 
                   row['code'] in stock_list and
                    prescreen(row['title']) and
                  await is_relevant(row["title"]) == "relevant"]
len(relevant_target_data_ytd)

0

In [15]:
relevant_target_data_ytd

[]

In [16]:
for row in relevant_target_data_ytd:
    row["summary"] = generate_summary(row["link"])

In [69]:
prompt = f"""
You are a financial news journalist tasked with creating a concise news-style summary of regulatory announcements from listed companies.

I have collected the following regulatory announcements:

{json.dumps(relevant_target_data_ytd, indent=2)}

Please create a concise news-style summary that:
1. Rewrites announcement titles into engaging, journalistic headlines (not just copying the legal titles)
2. Highlights the most significant and noteworthy announcements
3. Groups related announcements when appropriate
4. Includes the source (company name or link) for each item
5. Focuses on information that would be relevant to investors and stakeholders
6. Uses a news writing style - clear, engaging, and accessible

Format your response as:

## Regulatory Announcements Summary

- **News-style headline**: Brief summary of the key points. (Source: [company name or link])
- **News-style headline**: Brief summary of the key points. (Source: [company name or link])
...

Requirements:
- Transform legal/formal announcement titles into news-style headlines that are more readable and engaging
- Be concise (aim for 200-400 words total)
- Prioritize the most significant announcements (major corporate actions, financial results, regulatory changes, etc.)
- Each bullet point should include the source information
- If multiple announcements are from the same company, you may group them together
- Focus on actionable information and material developments
- Write headlines in a journalistic style - active voice, clear, and informative
- Example: Instead of "Company ABC - Annual Results Announcement", write something like "ABC Reports Strong Annual Earnings" or "ABC Posts Record Revenue for Fiscal Year"

Output only the summary, no additional commentary.
"""

In [70]:
response = await client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": prompt}
    ]
)

listed_co_news = response.choices[0].message.content.strip()

In [132]:
def send_regulatory_summary_email(title, summary_text, recipient_email="davidlau512@gmail.com"):
    # Sender information
    sender_email = "david@xplorehk.com"
    sender_name = "Global Headlines"
    
    # Convert markdown to HTML
    import markdown
    html_content = markdown.markdown(summary_text)
    
    # Email subject
    from datetime import datetime
    subject = f"{title} - {datetime.now().strftime('%B %d, %Y')}"
    
    # Create Mailjet client and send email
    try:
        api_key = os.environ.get('MJ_APIKEY_PUBLIC')
        api_secret = os.environ.get('MJ_APIKEY_PRIVATE')
        
        if not api_key or not api_secret:
            print("❌ Mailjet API keys not found in environment variables")
            return False
        
        mailjet = Client(auth=(api_key, api_secret), version='v3.1')
        
        data = {
            'Messages': [
                {
                    "From": {
                        "Email": sender_email,
                        "Name": sender_name
                    },
                    "To": [{"Email": recipient_email, "Name": "Recipient"}],
                    "Subject": subject,
                    "TextPart": summary_text,
                    "HTMLPart": html_content
                }
            ]
        }
        
        result = mailjet.send.create(data=data)
        
        if result.status_code == 200:
            print(f"✅ Corporate announcement summary sent successfully to {recipient_email}!")
            return True
        else:
            print(f"❌ Failed to send email. Status code: {result.status_code}")
            print(f"Response: {result.json()}")
            return False
    except Exception as e:
        print(f"❌ Failed to send email: {e}")
        return False

In [72]:
# Send the email
send_regulatory_summary_email("Corporate Announcement Summary", listed_co_news)

✅ Corporate announcement summary sent successfully to davidlau512@gmail.com!


True

In [73]:
options = webdriver.ChromeOptions()
options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
# These two lines help avoid detection and reduce cookie banner issues
options.add_argument('--disable-blink-features=AutomationControlled')
options.add_experimental_option("excludeSwitches", ["enable-automation"])

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
wait = WebDriverWait(driver, 20)
driver.get("http://www.aastocks.com/en/stocks/market/ipo/iponews.aspx")

In [74]:
driver.page_source

'<html xmlns="http://www.w3.org/1999/xhtml" xmlns:fb="http://www.facebook.com/2008/fbml" xmlns:og="http://ogp.me/ns#"><head id="Head1"><meta http-equiv="X-UA-Compatible" content="IE=Edge"><meta name="google-site-verification" content="PSvX40cckR7V_q8QVaRk5jnTEIeinakRTyMqcjv9WPI"> <script type="text/javascript" async="" src="https://www.googletagmanager.com/gtag/js?id=G-38RQTHE076&amp;l=_gtagL&amp;cx=c&amp;gtm=4e5ca1"></script><script type="text/javascript" async="" src="https://www.googletagmanager.com/gtag/js?id=G-FL2WFCGS0Y&amp;l=_gtagL"></script><script type="text/javascript" async="" src="https://ssl.google-analytics.com/ga.js"></script><script type="text/javascript">\nvar _gaq = _gaq || [];\n_gaq.push([\'_setAccount\', \'UA-20790503-3\']);\n_gaq.push([\'_setDomainName\', \'www.aastocks.com\']);\n_gaq.push([\'_setSampleRate\', \'5\']);\n_gaq.push([\'_trackPageview\']);\n_gaq.push([\'_trackPageLoadTime\']);\n_gaq.push([\'a3._setAccount\', \'UA-130882905-1\']);\n_gaq.push([\'a3._setD

In [120]:
import html  # This is the key import!

def extract_news_list(driver, timeout=20):
    news_items = []
    
    wait = WebDriverWait(driver, timeout)
    wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "div[ref^='NOW.']")))
    
    news_blocks = driver.find_elements(By.CSS_SELECTOR, "div[ref^='NOW.']")
    
    for block in news_blocks:
        item = {"title": "", "date": "", "content": "", "link": "", "image_url": None}
#         print(block.get_attribute("outerHTML"))
        
        try:
            # === TITLE: Get from title attribute AND unescape HTML entities ===
            title_link = block.find_element(By.CSS_SELECTOR, "a[id*='lnkNews_']:not([id*='lnkNewsImage'])")
            raw_title = title_link.get_attribute("title") or title_link.text.strip()
            if raw_title:
                # This converts &amp;lt; → <, &amp;gt; → >, &amp;#39; → ', etc.
                clean_title = html.unescape(raw_title)
                # Optional: remove the <IPO> tag if you don't want it
                clean_title = clean_title.replace("<IPO>", "").strip()
                item["title"] = clean_title
            else:
                item["title"] = "[No title]"
        except Exception as e:
            item["title"] = "[Title error]"
        
        try:
            item["link"] = title_link.get_attribute("href")
            if item["link"] and not item["link"].startswith("http"):
                item["link"] = "https://www.aastocks.com" + item["link"]
        except:
            pass
        
        # === DATE ===
        try:
            time_text = block.find_element(By.CSS_SELECTOR, ".newstime4 .inline_block").text.strip()
            # Extract YYYY/MM/DD part
            import re
            match = re.search(r'(\d{4}/\d{1,2}/\d{1,2})', time_text)
            if match:
                y, m, d = match.group(1).split("/")
                item["date"] = f"{int(d):02d}/{int(m):02d}/{y}"
            else:
                item["date"] = "[Unknown]"
        except:
            item["date"] = "[Unknown]"
        
        # === CONTENT ===
        try:
            content = block.find_element(By.CSS_SELECTOR, ".newscontent4")
            text = content.text.strip()
            footer = "~AAStocks Financial NewsWeb Site: www.aastocks.com"
            if text.endswith(footer):
                text = text[:-len(footer)].strip()
            item["content"] = text or "[No content]"
        except:
            item["content"] = "[No content]"
        
        # === IMAGE (optional) ===
        try:
            img = block.find_element(By.CSS_SELECTOR, ".newsImage4a img")
            item["image_url"] = img.get_attribute("src")
        except:
            pass
        
#         print(item)
        
        if item["title"] and item["title"] not in {"[No title]", "[Title error]"}:
            news_items.append(item)
    
    return news_items

In [121]:
news_list = extract_news_list(driver)

In [125]:
current_ipo_news = [news for news in news_list if news["date"] in data_range]
current_ipo_news

[{'title': 'GUOXIA TECH Closes Up 128.4% to $45.9 at Midday',
  'date': '16/12/2025',
  'content': 'On debut today, GUOXIA TECH(02655.HK) opened at $38, up 89.1% from the listing price of $20.1. Peaking/ bottoming at $46.8/37.84, the stock closed midday at $45.9, up 128.4% from the listing price of $20.1, on volume of 6.86 million shares and turnover of $268.28 million.',
  'link': 'https://www.aastocks.com/en/stocks/news/aafn-con/NOW.1490647/ipo-news/AAFN',
  'image_url': 'https://plib.aastocks.com/aafnnews/image/medialib/20191017105312335_s.jpg'},
 {'title': "Nearly Half of 15 Firms Invested in by C Capital's Fund Plan to Go Public Next Yr",
  'date': '16/12/2025',
  'content': 'C Capital, co-founded by former NEW WORLD DEV (00017.HK) CEO Adrian Cheng, has invested in around 15 companies through its blind pool fund.C Capital CEO Ben Cheng told Hong Kong media that seven of the abovementioned 15 companies are planning to go public next year in Hong Kong, Mainland China, or the US, inc

In [129]:
prompt = f"""
You are an expert financial news summarizer specializing in Hong Kong IPO and stock market updates.

Your task is to read the following list of raw news items (each with title, date, and content) and produce a concise daily bulletin in English.

Instructions:
- Cover EVERY news item in the list — do not skip any, even if it seems less significant.
- Identify the main company/stock (e.g., from stock code like (02655.HK) or name in title/content) for each item.
- If multiple items are about the SAME company/stock, group them under ONE bullet point and combine into a coherent summary (merge details chronologically or logically; do not list titles separately).
- If an item is about a unique company/stock (or general/non-company-specific), give it its own bullet point.
- For each bullet, start with the company/stock name/code prominently, then summarize in 1–2 sentences.
- Prioritize recent details within groups (use dates if relevant).
- Skip items with "[No content]" only if the title adds no value; otherwise, summarize the title alone.
- Sort the final bullets by recency (newest items/companies first).
- Do not include links, image URLs, or source mentions.

Output format: A list of bullet points only. No introduction, no conclusion.

Here is today's raw news data:

{json.dumps(current_ipo_news, indent=2)}

Now generate the summary bulletin.

"""

In [130]:
response = await client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": prompt}
    ]
)

ipo_news = response.choices[0].message.content.strip()

In [131]:
ipo_news

"- **GUOXIA TECH (02655.HK)**: On its debut on December 16, 2025, GUOXIA TECH opened at $38, up 89.1% from its listing price of $20.1, and peaked at $46.8 before closing midday at $45.9, marking a 128.4% increase. Prior to its official listing, the stock closed at $37.8 in the gray market on December 15, 2025, up 88.1% from the listing price.\n\n- **ZHIHUI MINING (02546.HK)**: The IPO of ZHIHUI MINING, a Tibet mining company, garnered significant interest as it recorded a margin oversubscription of approximately 3,355 times on December 16, 2025, with about HKD 184.58 billion received by brokers.\n\n- **C Capital**: As of December 16, 2025, nearly half of the 15 firms invested in by C Capital's fund plan to go public next year, including various companies with intentions to list in Hong Kong, Mainland China, or the US. C Capital was co-founded by Adrian Cheng, former CEO of NEW WORLD DEV (00017.HK)."

In [133]:
send_regulatory_summary_email("HK IPO news", ipo_news)

✅ Corporate announcement summary sent successfully to davidlau512@gmail.com!


True

In [78]:
news_blocks = driver.find_elements(By.CSS_SELECTOR, "div[ref^='NOW.']")

In [84]:
for block in news_blocks:
    item = {"title": "", "date": "", "content": ""}

    try:
        # 1. Title - inside <a> with id containing "lnkNews"
        title_elem = block.find_element(By.CSS_SELECTOR, "a[id*='lnkNews']")
        item["title"] = title_elem.text.strip()
        # Fallback if text is empty (sometimes title is in attribute)
        if not item["title"]:
            item["title"] = title_elem.get_attribute("title") or ""
        item["title"] = item["title"].replace("<IPO>", "").strip()  # clean tags if needed
    except:
        item["title"] = "[No title found]"

    try:
        # 2. Date - inside script with ConvertToLocalTime or visible text like "2025/12/16 12:15"
        time_divs = block.find_elements(By.CSS_SELECTOR, ".newstime4")
        date_text = ""
        for div in time_divs:
            text = div.text.strip()
            if not text:
                # Check inner script
                scripts = div.find_elements(By.TAG_NAME, "script")
                for s in scripts:
                    script_text = s.get_attribute("innerHTML")
                    match = date_pattern.search(script_text)
                    if match:
                        date_text = match.group(1)
                        break
            else:
                match = date_pattern.search(text)
                if match:
                    date_text = match.group(1)
                    break
            if date_text:
                break

        if date_text:
            # Convert "2025/12/16" → "16/12/2025"
            y, m, d = date_text.split("/")
            item["date"] = f"{int(d):02d}/{int(m):02d}/{y}"
        else:
            item["date"] = "[Unknown date]"
    except:
        item["date"] = "[Unknown date]"

    try:
        # 3. Content - in div with class "newscontent4"
        content_elem = block.find_element(By.CSS_SELECTOR, ".newscontent4")
        item["content"] = content_elem.text.strip()
        # Remove trailing "~AAStocks Financial News..." if present
        if item["content"].endswith("~AAStocks Financial NewsWeb Site: www.aastocks.com"):
            item["content"] = item["content"][:-len("~AAStocks Financial NewsWeb Site: www.aastocks.com")].strip()
    except:
        item["content"] = "[No content]"

    # Only add if we got at least a title
    if item["title"] and item["title"] != "[No title found]":
        news_items.append(item)

test
<selenium.webdriver.remote.webelement.WebElement (session="927e3d9fdcf20f05a4567c40cae1b4ea", element="f.3AA6C2CF3E2F3FE490D6AB2642E8C7FD.d.B8825C1235F4D835B911767AD91874AC.e.67")>


In [40]:
pdf_path = os.path.join(os.getcwd(), "2025121500019.pdf")

documents = SimpleDirectoryReader(
    input_files=[pdf_path],
).load_data()                

summary_index = SummaryIndex(documents)

summary_query_engine = summary_index.as_query_engine(
    response_mode="tree_summarize",
    use_async=True,
)    

In [41]:
prompt='''
You are an expert financial analyst specializing in IPO prospectuses. Summarize the provided document in a detailed, structured manner, focusing on the company's business, financials, offering details, and key parties. Use clear sections with bullet points for readability. Avoid legal jargon; explain concepts simply. If information is missing or unclear, note it explicitly.

Structure your summary as follows:

1. **Company Overview**: Describe the company's main business, industry, products/services, history, and competitive position. Include any key risks or growth strategies mentioned.

2. **Key Financials**: Summarize recent financial performance, including revenue, profit/loss, EBITDA (if applicable), balance sheet highlights (e.g., assets, liabilities, cash position), and any trends or projections. Cover the last 2-3 years if available.

3. **Offering Structure**: Detail the IPO mechanics, including:
   - Total number of shares offered (public offer vs. international placement).
   - Price range or final offer price.
   - Net proceeds and intended use (e.g., expansion, debt repayment).
   - Cornerstone investors (names, amounts committed, lock-up periods).

4. **Parties Involved**: List key external parties, including:
    - Sponsor, Overall Coordinator, Global Coordinator, Joint Bookrunner and Joint Lead Manager

Base your summary solely on the document content. If a section lacks data, state "Not specified in the prospectus." Keep the total response under 1,000 words, prioritizing accuracy and neutrality.
'''

response = summary_query_engine.query(prompt)
print(response)

BadRequestError: Error code: 400 - {'error': {'message': 'This endpoint\'s maximum context length is 128000 tokens. However, you requested about 150997 tokens (146997 of text input, 4000 in the output). Please reduce the length of either one, or use the "middle-out" transform to compress your prompt automatically.', 'code': 400, 'metadata': {'provider_name': None}}}

In [32]:
llm

OpenRouter(callback_manager=<llama_index.core.callbacks.base.CallbackManager object at 0x000001FF59A43D50>, system_prompt=None, messages_to_prompt=<function messages_to_prompt at 0x000001FF45F2CF40>, completion_to_prompt=<function default_completion_to_prompt at 0x000001FF46118220>, output_parser=None, pydantic_program_mode=<PydanticProgramMode.DEFAULT: 'default'>, query_wrapper_prompt=None, model='anthropic/claude-3-haiku', temperature=0.1, max_tokens=4000, logprobs=None, top_logprobs=0, additional_kwargs={}, max_retries=5, timeout=900.0, default_headers=None, reuse_client=True, api_key='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', api_base='https://openrouter.ai/api/v1', api_version='', strict=False, reasoning_effort=None, modalities=None, audio_config=None, context_window=3900, is_chat_model=True, is_function_calling_model=False, tokenizer=None)

In [2]:
import yfinance as yf

def get_price_stats(ticker: str) -> dict:
    """
    Returns:
      - current_price
      - last_close (last trading day's close)
      - ytd_close (last trading close of previous calendar year)
      - pct_change_ytd (from ytd_close to current)
    """
    t = yf.Ticker(ticker)

    # Current price (fallbacks included)
    info = getattr(t, "fast_info", {}) or {}
    current = info.get("last_price") or info.get("regularMarketPrice")
    if current is None:
        current = t.history(period="1d")["Close"].iloc[-1]

    # Last trading day's close
    last_close = info.get("previous_close") or info.get("regularMarketPreviousClose")
    if last_close is None:
        last_close = t.history(period="5d")["Close"].dropna().iloc[-1]

    # YTD close = last trading close of previous calendar year
    hist = t.history(period="max")
    if hist.empty:
        raise ValueError(f"No price history for {ticker}")

    current_year = hist.index[-1].year
    prev_year = hist[hist.index.year < current_year]
    if prev_year.empty:
        raise ValueError(f"Not enough history to compute ytd_close for {ticker}")

    ytd_close = prev_year["Close"].iloc[-1]
    pct_change_ytd = (current / ytd_close - 1.0) * 100.0

    return {
        "ticker": ticker,
        "current_price": float(current),
        "last_close": float(last_close),
        "ytd_close": float(ytd_close),
        "pct_change_ytd": float(pct_change_ytd),
    }

In [3]:
def _safe_float(x):
    try:
        if x is None:
            return None
        return float(x)
    except Exception:
        return None

def _latest_close(ticker):
    try:
        s = ticker.history(period="5d")["Close"].dropna()
        if s.empty:
            return None
        return float(s.iloc[-1])
    except Exception:
        return None


def _previous_close(ticker):
    info = getattr(ticker, "fast_info", {}) or {}
    val = info.get("previous_close") or info.get("regularMarketPreviousClose")
    v = _safe_float(val)
    if v is not None:
        return v
    # Fallback: last available close (can be same as "current" if market is closed)
    return _latest_close(ticker)

In [4]:
ticker = yf.Ticker("^HSI")
# _previous_close(ticker)

In [5]:
info = getattr(ticker, "fast_info", {})
info

lazy-loading dict with keys = ['currency', 'dayHigh', 'dayLow', 'exchange', 'fiftyDayAverage', 'lastPrice', 'lastVolume', 'marketCap', 'open', 'previousClose', 'quoteType', 'regularMarketPreviousClose', 'shares', 'tenDayAverageVolume', 'threeMonthAverageVolume', 'timezone', 'twoHundredDayAverage', 'yearChange', 'yearHigh', 'yearLow']

In [6]:
ticker.history(period="2d")

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2026-01-21 00:00:00+08:00,26397.039062,26692.779297,26397.039062,26585.060547,2998700000,0.0,0.0
2026-01-22 00:00:00+08:00,26750.509766,26779.220703,26499.140625,26541.720703,0,0.0,0.0


In [31]:
async def fetch_article_html(url):
    """Fetch the HTML content of an article page"""
    try:
        html_dict = extract_html.get_raw_html(url)
        if html_dict and html_dict.get("html"):
            return html_dict["html"]
        return None
    except Exception as e:
        print(f"    WARNING: Error fetching article HTML: {e}")
        return None

In [32]:
html_content = await fetch_article_html("https://news.futunn.com/en/post/66803841/ipo-news-minimax-a-globally-leading-ai-large-model-company?level=1&data_ticket=1767075888981595")

In [36]:
clean_text = remove_html_tags(html_content)
if len(clean_text) > 200000:
    clean_text = clean_text[:200000] + "..."

In [41]:
clean_text.replace("\n", "").replace("  ", " ")

'  IPO News | MiniMax, a globally leading AI large model company, has launched its initial public offering with cornerstone investors including Alibaba and Abu Dhabi. The minimum subscription amount is HKD 3,333.28.        Investing                 Stocks                    HK Stocks                     US Stocks                     JP Stocks                     A-Shares                     Margin Trading                     IPO                     Invest Regularly                    Derivatives & Cryptos                    ETF                     Options                     Futures                     Crypto                     Futu Money Plus                    Cash Plus                     Funds                     US Treasuries                     Structured Products                    Futu PWM                    Private Wealth Management                     Trust Services                  Go FUTU Securities check more                >               Markets                 Quotes  

In [25]:
async def is_article_from_today(client, clean_text, url):
    
    today = datetime.now()
    today_str = today.strftime("%B %d, %Y")  # e.g., "December 29, 2025"
    today_str_alt = today.strftime("%d %B %Y")  # e.g., "29 December 2025"
    today_str_short = today.strftime("%m/%d/%Y")  # e.g., "12/29/2025"
    
    prompt = f"""
You are analyzing a news article to determine if it was published today.

Today's date is: {today_str} (also known as {today_str_alt} or {today_str_short})

Here is the article content (extracted from HTML):
{clean_text}

Article URL: {url}

Look for publication date information in the article. This could be:
- A "Published" or "Updated" date
- A timestamp
- A date in the article metadata
- Any date information visible on the page

Determine if this article was published TODAY ({today_str}).

Return ONLY "YES" if the article is from today, or "NO" if it's from a different date.
If you cannot determine the date, return "NO".
"""
    
    try:
        response = await client.chat.completions.create(
            model="openai/gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1,
            max_tokens=10,
        )
        
        result = response.choices[0].message.content.strip().upper()
        return result == "YES"
    except Exception as e:
        print(f"    WARNING: Error checking article date: {e}")
        return False  # Default to False if check fails

In [39]:
await is_article_from_today(client, clean_text.replace("\n", ""), "https://news.futunn.com/en/post/66790061/the-us-materials-sector-emerges-as-the-hidden-winner-of?chain_id=R4cqdOkIUz1fSH.1kl93t6&futusource=news_headline_list&global_content=%7B%22promote_id%22%3A13766%2C%22sub_promote_id%22%3A28%2C%22f%22%3A%22news.futunn.com%2Fen%2Fmain%22%7D&report_id=301133&report_type=market&src=3%2C3")

True